# Trabalho final: Modelos de recomendação e análise de espaços latentes

**Disciplina:** *Aprendizado não-supervisionado* - *MC847*\
**Instituição:** *Universidade Estadual de Campinas*\
**Professor(a):** *Marcos Medeiros Raimundo*\
**Semestre:** *2s2026*

| Nome | RA |
|---|---|
| *Integrante 1* | *RA 1* |
| *Integrante 2* | *RA 2* |
| *Integrante 3 (opcional)* | *RA 3* |

Preencha a tabela acima com os nomes e RAs dos integrantes da equipe (2 ou 3 alunos, impreterivelmente).

## Introdução

Em sistemas de recomendação, a principal tarefa é aprender a preferência de um usuário a partir de um histórico parcial de interações.
Isso pode ser visto como um problema de *completar informação*: dado o histórico parcial de um usuário, quais itens provavelmente faltam?
Neste trabalho vocês vão prever notas não observadas com modelos que fazem suposições diferentes sobre essa estrutura e, depois, investigar o que o espaço aprendido contém e quais suposições o produziram.

O trabalho é dividido em três etapas, sendo as duas primeiras comuns a todos os grupos e a terceira a ser definida por cada grupo.
A primeira etapa consiste em definir uma baseline que será usada como referência para as demais etapas.
A segunda consiste em realizar e analisar experimentos para entender como diferentes suposições sobre a estrutura latente dos dados afetam o desempenho do modelo.
A última etapa é a contribuição de cada grupo para o problema, que pode ser uma proposta de melhoria do modelo ou uma análise mais profunda de algum aspecto do problema.

As duas primeiras etapas têm códigos de referência, que podem ser usados como base para a implementação dos modelos.
Vocês podem usar bibliotecas externas com implementações de modelos, mas devem deixar claro quais bibliotecas foram usadas e como elas foram usadas.
Por exemplo, vocês podem reescrever o modelo de fatoração de matrizes em PyTorch ou Jax para melhorar a performance, desde que justifiquem a escolha e expliquem como a implementação foi mantida fiel ao modelo original.

## Dataset

Utilizaremos o conjunto de dados MovieLens 100K [(Harper e Konstan, 2015)](https://www.scirp.org/reference/referencespapers?referenceid=3586299): 100 mil notas inteiras de 1 a 5, dadas por 943 usuários a 1.682 filmes, com pelo menos 20 notas por usuário.
A matriz tem densidade de cerca de 6%, cada usuário tem idade (em anos), gênero e ocupação (21 categorias) e cada filme tem título, ano e 19 indicadores de gênero, e um filme pode ter vários gêneros.

Devido Ã  natureza do problema, a ausÃªncia de notas nÃ£o pode ser tratada como aleatÃ³ria: Os motivos pelos quais um usuÃ¡rio nÃ£o avaliou um item podem ser variados, incluindo falta de interesse, falta de exposiÃ§Ã£o ao item ou simplesmente esquecimento.
Isso significa que os dados de treinamento são enviesados e não representam a distribuição real de preferências dos usuários.

In [ ]:
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from numpy.typing import NDArray

sns.set_theme(style="whitegrid")

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MOVIELENS_URL = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"
DATA_DIR = "./data/"

def set_seed(seed: int) -> None:
    """Set the random seed for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

In [ ]:
from pandas.core.frame import DataFrame

GENRES = [
    "unknown", "Action", "Adventure", "Animation", "Children's", "Comedy",
    "Crime", "Documentary", "Drama", "Fantasy", "Film-Noir", "Horror",
    "Musical", "Mystery", "Romance", "Sci-Fi", "Thriller", "War", "Western"
]


def load_movielens() -> tuple[DataFrame, DataFrame, DataFrame]:
    """Download and extract the MovieLens dataset."""
    import zipfile

    import requests

    data_dir = Path(DATA_DIR)
    data_dir.mkdir(parents=True, exist_ok=True)

    zip_path = data_dir / "ml-100k.zip"
    target_dir = data_dir / "ml-100k"
    
    if not target_dir.exists():
        if not zip_path.exists():
            print("Downloading MovieLens dataset...")
            response = requests.get(MOVIELENS_URL)

            with open(zip_path, "wb") as f:
                f.write(response.content)

            print("Download complete.")

        print("Extracting MovieLens dataset...")
        with zipfile.ZipFile(zip_path) as zip_ref:
            zip_ref.extractall(data_dir)

        print(f"Extracted to {data_dir}.")

    ratings = pd.read_csv(
            target_dir / "u.data",
            sep="\t",
            names=["user_id", "item_id", "rating", "timestamp"],
            dtype={"user_id": np.int32, "item_id": np.int32, "rating": np.float32, "timestamp": np.int64},
        )

    users = pd.read_csv(
            target_dir / "u.user",
            sep="|",
            names=["user_id", "age", "gender", "occupation", "zip_code"],
            dtype={"user_id": np.int32, "age": np.int32, "gender": str, "occupation": str, "zip_code": str},
        )

    item_cols = ["item_id", "title", "release_date", "video_release_date", "imdb_url", *GENRES]

    movies = pd.read_csv(
        target_dir / "u.item",
        sep="|",
        encoding="latin-1",
        names=item_cols,
        usecols=["item_id", "title", "release_date", *GENRES],
        dtype={"item_id": np.int32, "title": str, "release_date": str},
    )

    movies["year"] = pd.to_numeric(movies["title"].str.extract(r"\((\d{4})\)")[0], errors="coerce")

    movies["genres"] = movies[GENRES].apply(
        lambda row: "|".join([g for g in GENRES if row[g] == 1]), axis=1
    )

    # Normalização para que os IDs de usuário e item comecem em 0
    ratings["user_id"] -= 1
    ratings["item_id"] -= 1
    users["user_id"] -= 1
    movies["item_id"] -= 1

    return ratings, users, movies


def ratings_to_matrix(ratings: pd.DataFrame, fill_value: float = float("nan")) -> NDArray[np.float64]:
    """Convert ratings DataFrame to a user-item matrix."""
    num_users = ratings["user_id"].max() + 1
    num_items = ratings["item_id"].max() + 1

    user_item_matrix = np.full((num_users, num_items), fill_value, dtype=np.float64)
    user_item_matrix[ratings["user_id"], ratings["item_id"]] = ratings["rating"].values.astype(np.float64)

    return user_item_matrix


In [ ]:
ratings, users, movies = load_movielens()
density = len(ratings) / (len(users) * len(movies))
print(f"notas={len(ratings):,}; usuários={len(users):,}; filmes={len(movies):,}; densidade={density:.3f}")

## Formulação do problema

Uma suposição comum em sistemas de recomendação é que a preferência de um usuário por um item pode ser representada como uma função sobre um espaço latente compartilhado entre usuários e itens.
Ou seja, considerando que cada usuário e cada item podem ser representados como vetores em um espaço latente, a nota de um usuário para um item pode ser modelada como uma função que mede a compatibilidade entre os vetores do usuário e do item.

Utilizaremos uma formulação geral para este problema, que pode acomodar diferentes modelos e suposições sobre como os vetores latentes são gerados e como a função de compatibilidade é definida.
Considere um conjunto de usuários $U$ e um conjunto de itens $I$, e seja $R$ a matriz de notas, a nota do usuário $u$ para o item $i$, $R_{ui}$ é modelada como:

$$
R_{ui} = \mu + b_u + b_i + f(\mathbf{z}_u, \mathbf{w}_i) + \epsilon_{ui}
$$
onde $\mu$ é a média global das notas, $b_u$ e $b_i$ são os *biases* do usuário e do item, respectivamente, $\mathbf{z}_u$ e $\mathbf{w}_i$ são os vetores latentes do usuário e do item, $f$ é a função de compatibilidade entre os vetores latentes e $\epsilon_{ui}$ é o erro de modelagem para aquele par usuário-item.

Um desafio importante é que a matriz de notas $R$ é muito esparsa, ou seja, grande parte dos pares usuário-item não possuem notas observadas. Isso significa que aprender a estrutura latente compartilhada pode ser difícil sem fazer suposições adicionais sobre como os vetores latentes são gerados e como a função de compatibilidade é definida.

Vamos explorar como diferentes suposições afetam o desempenho do modelo e o que podemos aprender sobre a estrutura latente dos dados através da inspeção do espaço latente aprendido.

> **Importante:** Os modelos têm uma abordagem transdutiva, ou seja, eles aprendem os vetores latentes apenas para os usuários e itens presentes no conjunto de treinamento.
As features de um usuário não são as notas que ele deu, mas sim apenas um identificador único, e o mesmo vale para os itens.
Isso permite que o modelo possua parâmetros específicos para cada usuário, mas também significa que ele não pode generalizar para novos usuários ou itens sem re-treinamento.

## Diretrizes para o trabalho

Para cada experimento, sigam as seguintes diretrizes:

> 1. Escrevam a hipótese e a previsão **antes** de rodar cada experimento.
> 2. Resultados negativos ou inconclusivos valem, se vierem com **análise crítica**.
> 3. Todos os modelos, inclusive os mais simples, passam pelo mesmo protocolo (a seguir).
> 4. Separem o que um resultado diz sobre a previsão do que ele diz sobre a representação latente.
> 5. Citem fontes sempre que necessário, seja para motivar uma escolha, ou para justificar uma decisão.

A nota não depende da métrica obtida pelo modelo, mas da qualidade das hipóteses, aplicação correta do protocolo e da análise das evidências a favor, contra ou insuficientes.

### Protocolo experimental

Os experimentos devem ser controlados, com uma variável de interesse por vez e com o mesmo protocolo:

**Split dos dados.** O conjunto de dados deve ser dividido em treino e teste, com 80% das notas para treino e 20% para teste.
A divisão deve ser feita de forma estratificada por usuário, garantindo que cada usuário tenha pelo menos uma nota em cada conjunto, e todo filme deve ter pelo menos uma nota no conjunto de treino.

**Métricas.** RMSE (principal) e MAE, com previsões truncadas em $[1,5]$, e NLL quando o modelo define uma verossimilhança comparável.

**Seleção de hiperparâmetros.** O conjunto de teste é usado uma única vez, para reportar o desempenho final.
Os hiperparâmetros (dimensão latente, regularização, variância do ruído, número de épocas e assim por diante) são escolhidos por *cross-validation* com $K$ folds dentro do conjunto de treino, respeitando as mesmas restrições da divisão de treino e teste.
Depois da escolha, o modelo é treinado com todo o treino e avaliado no teste.

**Incerteza.** Compare resultados com **intervalos de confiança** sobre as métricas.
Vocês podem usar *cross-validation*, reportando a média e o desvio padrão das métricas entre os folds, que refletem a variação causada pelos dados de treino e pela inicialização, ou *bootstrapping*, amostrando com reposição o conjunto de teste e reportando a média e o intervalo de percentis de 95%, que reflete a variação da amostra de teste.

In [ ]:
def split_ratings(
    ratings: pd.DataFrame,
    test_frac: float = 0.2,
    seed: int = 0,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    df = (
        ratings.sample(frac=1, random_state=seed)
               .sort_values("user_id", kind="stable")
               .reset_index(drop=True)
    )

    grouped = df.groupby("user_id")
    pos = (
        grouped.cumcount() /
        grouped["user_id"].transform("count")
    ).to_numpy()

    splits = np.zeros(len(df), dtype=np.int8)
    splits[pos >= (1 - test_frac)] = 1

    # Trocar a primeira aparição de cada filme para o conjunto de treino
    first_appearance = df.groupby("item_id").head(1).index
    splits[first_appearance] = 0

    train_df = df.iloc[splits == 0]
    test_df = df.iloc[splits == 1]

    return train_df, test_df

In [ ]:
from typing import Protocol

from sklearn.decomposition import PCA
from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
)

# Funções auxiliares

def clip_ratings(prediction: NDArray[np.float64]) -> NDArray[np.float64]:
    return np.clip(np.asarray(prediction, dtype=np.float64), 1.0, 5.0)


class Model(Protocol):
    def fit(self, train_df: DataFrame) -> None: ...

    def predict(
        self, user_id: NDArray[np.intp], item_id: NDArray[np.intp],
    ) -> NDArray[np.float64]: ...


class ProbabilisticModel(Model, Protocol):
    def predict_log_proba(
        self,
        user_id: NDArray[np.intp],
        item_id: NDArray[np.intp],
        ratings: NDArray[np.float64],
    ) -> NDArray[np.float64]: ...


def evaluate_model(
    model: Model | ProbabilisticModel, **dfs: DataFrame
) -> dict[str, float]:
    metrics: dict[str, float] = {}
    
    for name, df in dfs.items():
        users = df["user_id"].to_numpy(dtype=np.intp)
        items = df["item_id"].to_numpy(dtype=np.intp)
        ratings = df["rating"].to_numpy(dtype=np.float64)

        preds = model.predict(users, items)
        preds_clipped = clip_ratings(preds)

        metrics |= {
            f"{name}_rmse": root_mean_squared_error(ratings, preds_clipped),
            f"{name}_mae": mean_absolute_error(ratings, preds_clipped),
            # Vocês podem adicionar mais métricas aqui
        }

        if hasattr(model, "predict_log_proba"):
            log_proba = model.predict_log_proba(users, items, ratings)
            metrics[f"{name}_nll"] = float(-np.mean(log_proba))

    return metrics



In [ ]:
train_df, test_df = split_ratings(ratings, seed=SEED)

In [ ]:
# ===== Utilidades do protocolo experimental (comuns a todos os modelos) =====
import time
from itertools import product

from scipy.stats import spearmanr
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression, RidgeCV
from sklearn.manifold import TSNE
from sklearn.metrics import adjusted_rand_score
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

FIG_DIR, RES_DIR = Path("figs"), Path("results")
FIG_DIR.mkdir(exist_ok=True)
RES_DIR.mkdir(exist_ok=True)

K_FOLDS = 5      # folds da validação cruzada (dentro do treino)
N_BOOT = 1000    # reamostragens bootstrap para os intervalos de confiança
N_USERS = int(train_df["user_id"].max()) + 1
N_ITEMS = int(train_df["item_id"].max()) + 1


def make_folds(df: pd.DataFrame, n_folds: int = K_FOLDS, seed: int = 0):
    """K folds disjuntos, estratificados por usuário (cada usuário distribui suas notas entre os folds).
    Mesma regra da divisão treino/teste: todo filme precisa ter ao menos uma nota no treino do fold.
    Se a nota de validação for a única do filme no treino do fold, ela é movida para o treino
    (e deixa de ser usada na validação)."""
    df = df.sample(frac=1, random_state=seed).reset_index(drop=True)
    fold = (df.groupby("user_id").cumcount() % n_folds).to_numpy()
    folds = []
    for f in range(n_folds):
        tr, va = df[fold != f], df[fold == f]
        orphan = ~va["item_id"].isin(tr["item_id"].unique())
        folds.append((pd.concat([tr, va[orphan]]), va[~orphan]))
    return folds


FOLDS = make_folds(train_df, K_FOLDS, seed=SEED)


def cv_scores(make_model, folds=FOLDS, **eval_kwargs) -> pd.DataFrame:
    """Treina um modelo novo em cada fold e avalia na validação do fold."""
    rows = []
    for fold_id, (tr, va) in enumerate(folds):
        set_seed(SEED + fold_id)
        model = make_model()
        t0 = time.perf_counter()
        model.fit(tr)
        fit_s = time.perf_counter() - t0
        rows.append({"fold": fold_id, "fit_s": fit_s, **evaluate_model(model, val=va)})
    return pd.DataFrame(rows)


def grid_search(model_cls, grid: dict, fixed: dict | None = None, folds=FOLDS, verbose=True) -> pd.DataFrame:
    """Busca em grade com validação cruzada. Retorna uma linha por configuração, com
    média e desvio padrão (entre folds) das métricas de validação."""
    fixed = fixed or {}
    keys = list(grid)
    rows = []
    for values in product(*grid.values()):
        params = dict(zip(keys, values))
        res = cv_scores(lambda: model_cls(**fixed, **params), folds)
        row = {**params, "fit_s": res["fit_s"].mean()}
        for col in res.columns:
            if col.startswith("val_"):
                row[col], row[col + "_std"] = res[col].mean(), res[col].std(ddof=1)
        rows.append(row)
        if verbose:
            print(params, f"RMSE={row['val_rmse']:.4f}±{row['val_rmse_std']:.4f}", flush=True)
    return pd.DataFrame(rows)


def best_row(results: pd.DataFrame, metric: str = "val_rmse") -> pd.Series:
    return results.loc[results[metric].idxmin()]


def boot_ci(values, transform=lambda x: x, n_boot: int = N_BOOT, seed: int = 0):
    """Média (após `transform`) e intervalo de percentis de 95% por bootstrap."""
    values = np.asarray(values, dtype=np.float64)
    rng = np.random.default_rng(seed)
    n = len(values)
    stats = np.array([values[rng.integers(0, n, n)].mean() for _ in range(n_boot)])
    return float(transform(values.mean())), float(transform(np.percentile(stats, 2.5))), float(transform(np.percentile(stats, 97.5)))


# ---------- Métricas de lista ----------
def score_matrix(model, n_users: int = N_USERS, n_items: int = N_ITEMS, block: int = 64) -> NDArray[np.float64]:
    """Nota prevista para todos os pares (usuário, filme), calculada em blocos de usuários."""
    scores = np.empty((n_users, n_items))
    all_items = np.arange(n_items)
    for start in range(0, n_users, block):
        users = np.arange(start, min(start + block, n_users))
        pred = model.predict(np.repeat(users, n_items), np.tile(all_items, len(users)))
        scores[users] = pred.reshape(len(users), n_items)
    return scores


def ranking_eval(scores, train_df, test_df, k: int = 10, threshold: float = 4.0) -> dict[str, NDArray]:
    """Métricas @k por usuário.
    Candidatos: filmes que o usuário NÃO avaliou no treino.
    Acerto: filme do teste com nota >= `threshold` presente na lista top-k.
    Só entram usuários com ao menos um filme relevante no teste."""
    scores = scores.copy()
    scores[train_df["user_id"].to_numpy(), train_df["item_id"].to_numpy()] = -np.inf
    top = np.argpartition(-scores, k, axis=1)[:, :k]
    top = np.take_along_axis(top, np.argsort(-np.take_along_axis(scores, top, 1), axis=1), 1)

    relevant = np.zeros(scores.shape, dtype=bool)
    good = test_df["rating"].to_numpy() >= threshold
    relevant[test_df["user_id"].to_numpy()[good], test_df["item_id"].to_numpy()[good]] = True

    hits = np.take_along_axis(relevant, top, 1)
    n_rel = relevant.sum(1)
    discount = 1.0 / np.log2(np.arange(2, k + 2))
    ideal = np.cumsum(discount)[np.minimum(n_rel, k).clip(min=1) - 1]
    valid = n_rel > 0
    return {
        f"precision@{k}": (hits.sum(1) / k)[valid],
        f"hit_rate@{k}": (hits.sum(1) > 0).astype(float)[valid],
        f"ndcg@{k}": ((hits * discount).sum(1) / ideal)[valid],
    }


class PopularityModel:
    """Referência para listas: recomenda os filmes mais avaliados no treino (não prevê nota)."""

    def fit(self, train_df: pd.DataFrame) -> None:
        self.counts = np.bincount(train_df["item_id"].to_numpy(), minlength=N_ITEMS).astype(np.float64)

    def predict(self, user_id, item_id):
        return self.counts[item_id]


class GlobalMean:
    """Previsão constante: a média global das notas de treino."""

    def fit(self, train_df: pd.DataFrame) -> None:
        self.mean = float(train_df["rating"].mean())

    def predict(self, user_id, item_id):
        return np.full(len(item_id), self.mean)


FINAL: dict[str, dict] = {}   # resultados finais no TESTE (cada modelo é avaliado uma única vez)
TRAINED: dict[str, object] = {}


def final_eval(name: str, model, train_df, test_df, rating_metrics: bool = True) -> dict:
    """Treina com todo o treino e avalia UMA vez no teste: RMSE, MAE, NLL (quando existir),
    métricas de lista e tempos. Intervalos de confiança por bootstrap do teste (95%)."""
    set_seed(SEED)
    t0 = time.perf_counter()
    model.fit(train_df)
    out = {"model": name, "fit_s": time.perf_counter() - t0}

    users = test_df["user_id"].to_numpy(dtype=np.intp)
    items = test_df["item_id"].to_numpy(dtype=np.intp)
    y = test_df["rating"].to_numpy(dtype=np.float64)

    t0 = time.perf_counter()
    pred = clip_ratings(model.predict(users, items))
    out["predict_s"] = time.perf_counter() - t0

    if rating_metrics:
        out["rmse"] = boot_ci((y - pred) ** 2, np.sqrt)
        out["mae"] = boot_ci(np.abs(y - pred))
    if hasattr(model, "predict_log_proba"):
        out["nll"] = boot_ci(-model.predict_log_proba(users, items, y))

    ranking = ranking_eval(score_matrix(model), train_df, test_df)
    for metric, per_user in ranking.items():
        out[metric] = boot_ci(per_user)

    FINAL[name], TRAINED[name] = out, model
    return out


def fmt_ci(ci) -> str:
    return f"{ci[0]:.4f} [{ci[1]:.4f}, {ci[2]:.4f}]" if isinstance(ci, tuple) else "-"


def final_table() -> pd.DataFrame:
    cols = ["rmse", "mae", "nll", "precision@10", "hit_rate@10", "ndcg@10"]
    rows = []
    for name, res in FINAL.items():
        rows.append({"modelo": name, **{c: fmt_ci(res.get(c)) for c in cols},
                     "treino (s)": round(res["fit_s"], 2), "inferência (s)": round(res["predict_s"], 3)})
    return pd.DataFrame(rows).set_index("modelo")


def plot_cv_curve(results: pd.DataFrame, x: str, hue: str | None = None, metric: str = "val_rmse", ax=None, logx=False):
    """RMSE de validação (média ± desvio entre folds) em função de um hiperparâmetro."""
    ax = ax or plt.gca()
    groups = results.groupby(hue) if hue else [(None, results)]
    for key, grp in groups:
        grp = grp.sort_values(x)
        ax.errorbar(grp[x], grp[metric], yerr=grp[metric + "_std"], marker="o", capsize=3,
                    label=None if key is None else f"{hue}={key}")
    if logx:
        ax.set_xscale("log")
    ax.set_xlabel(x)
    ax.set_ylabel(metric)
    if hue:
        ax.legend(fontsize=8)
    return ax


BEST: dict[str, dict] = {}   # hiperparâmetros escolhidos por validação cruzada


def rmse_by_item_popularity(model, train_df, test_df, edges=(0, 10, 50, 150, np.inf)) -> pd.Series:
    """RMSE no teste por faixa de popularidade do filme (nº de notas no treino)."""
    counts = np.bincount(train_df["item_id"].to_numpy(), minlength=N_ITEMS)
    items = test_df["item_id"].to_numpy(dtype=np.intp)
    users = test_df["user_id"].to_numpy(dtype=np.intp)
    err = (test_df["rating"].to_numpy() - clip_ratings(model.predict(users, items))) ** 2
    bins = pd.cut(counts[items], bins=list(edges), right=False,
                  labels=[f"{int(a)}–{int(b) - 1}" if np.isfinite(b) else f"≥{int(a)}" for a, b in zip(edges[:-1], edges[1:])])
    return pd.Series(err).groupby(bins, observed=True).mean().pow(0.5)

In [ ]:
# Exploração dos dados: distribuição das notas, esparsidade, sinais de ausência não aleatória e metadados
item_counts_all = ratings.groupby("item_id").size().reindex(range(len(movies)), fill_value=0)
user_counts_all = ratings.groupby("user_id").size()
item_mean_all = ratings.groupby("item_id")["rating"].mean()

print(f"notas={len(ratings):,} | usuários={len(users):,} | filmes={len(movies):,} | densidade={density:.2%}")
print(f"nota média={ratings['rating'].mean():.3f} | desvio={ratings['rating'].std():.3f}")
print(f"notas por usuário: min={user_counts_all.min()}, mediana={user_counts_all.median():.0f}, máx={user_counts_all.max()}")
print(f"notas por filme: min={item_counts_all.min()}, mediana={item_counts_all.median():.0f}, máx={item_counts_all.max()}")
top10 = item_counts_all.sort_values(ascending=False).head(int(0.1 * len(movies))).sum() / len(ratings)
print(f"10% dos filmes mais avaliados concentram {top10:.1%} das notas")
print(f"filmes com <=5 notas: {(item_counts_all <= 5).mean():.1%}")
print(f"treino={len(train_df):,} | teste={len(test_df):,}")

fig, axes = plt.subplots(2, 4, figsize=(20, 8))
sns.countplot(x="rating", data=ratings, ax=axes[0, 0], color="C0")
axes[0, 0].set_title("Distribuição das notas")

axes[0, 1].hist(user_counts_all, bins=40, color="C1")
axes[0, 1].set_title("Notas por usuário"); axes[0, 1].set_xlabel("nº de notas")

axes[0, 2].plot(np.sort(item_counts_all.to_numpy())[::-1], color="C2")
axes[0, 2].set_yscale("log"); axes[0, 2].set_title("Notas por filme (cauda longa)"); axes[0, 2].set_xlabel("posto do filme")

axes[0, 3].scatter(item_counts_all[item_mean_all.index], item_mean_all, s=6, alpha=0.5)
axes[0, 3].set_xscale("log"); axes[0, 3].set_title("Nota média × popularidade")
axes[0, 3].set_xlabel("nº de notas (log)"); axes[0, 3].set_ylabel("nota média")

axes[1, 0].hist(users["age"], bins=25, color="C3"); axes[1, 0].set_title("Idade dos usuários")
users["gender"].value_counts().plot.bar(ax=axes[1, 1], color="C4"); axes[1, 1].set_title("Gênero")
users["occupation"].value_counts().plot.barh(ax=axes[1, 2], color="C5"); axes[1, 2].set_title("Ocupação")
axes[1, 2].tick_params(labelsize=7)
movies[GENRES].sum().sort_values().plot.barh(ax=axes[1, 3], color="C6"); axes[1, 3].set_title("Filmes por gênero")
axes[1, 3].tick_params(labelsize=7)
plt.tight_layout(); plt.savefig(FIG_DIR / "eda_visao_geral.png", dpi=150); plt.show()

# Sinais de ausência não aleatória: quem avalia mais e o que é mais avaliado também é avaliado de forma diferente
pop_bins = pd.qcut(item_counts_all.rank(method="first"), 5, labels=["Q1 (nicho)", "Q2", "Q3", "Q4", "Q5 (populares)"])
mean_by_pop = ratings.assign(pop=pop_bins.reindex(ratings["item_id"]).to_numpy()).groupby("pop", observed=True)["rating"].agg(["mean", "count"])
print("\nNota média por quintil de popularidade do filme:\n", mean_by_pop.round(3))
act_bins = pd.qcut(user_counts_all.rank(method="first"), 5, labels=["U1 (poucas)", "U2", "U3", "U4", "U5 (muitas)"])
mean_by_act = ratings.assign(act=act_bins.reindex(ratings["user_id"]).to_numpy()).groupby("act", observed=True)["rating"].agg(["mean", "count"])
print("\nNota média por quintil de atividade do usuário:\n", mean_by_act.round(3))
print("\nPerfil: idade média=%.1f | %% mulheres=%.1f%% | filmes sem ano: %d" % (
    users["age"].mean(), 100 * (users["gender"] == "F").mean(), movies["year"].isna().sum()))

## Etapa 1: Baselines

O primeiro passo para entender o problema é estabelecer uma baseline, um modelo simples que sirva como referência da performance que podemos esperar de um modelo mais sofisticado.
Podemos simplificar a formulação geral do problema, considerando apenas a média global das notas e os *biases* do usuário e do item, sem considerar os vetores latentes:

$$R_{ui} = \mu + b_u + b_i + \epsilon_{ui}$$

Como o modelo remove a interação entre usuários e itens, ele não é capaz de capturar a estrutura latente compartilhada entre eles, mas ainda assim pode fornecer uma boa referência para avaliar o desempenho de modelos mais complexos.
[Koren, Y. et. al.](https://ieeexplore.ieee.org/document/5197422) discute este exato modelo como uma aproximação de primeira ordem para o problema, que pode ser melhorada com a adição de termos de segunda ordem, como os vetores latentes introduzidos na formulação geral.
Recomenda-se a leitura do artigo para entender melhor a motivação e interpretação do problema e as soluções propostas.

Devido Ã  linealidade do modelo,podemos aprender os *biases* do usuÃ¡rio e do item de forma eficiente, mesmo com a matriz de notas esparsa, atravÃ©s de regressÃ£o linear regularizada.

> **Tarefa 1.1**
> Treine o modelo de baseline e avalie seu desempenho, reportando RMSE e MAE no conjunto de teste.
> Lembre-se de usar o mesmo protocolo experimental para todos os modelos para encontrar o melhor conjunto de hiperparâmetros.
> Após treinar o modelo, analise os *biases* aprendidos e discuta o que eles podem nos dizer sobre a preferência dos usuários e a popularidade dos itens.

In [ ]:
class BaselineModel:
    def __init__(
        self,
        user_reg: float = 10.0,
        item_reg: float = 10.0,
        n_epochs: int = 20,
    ) -> None:
        self.user_reg = user_reg
        self.item_reg = item_reg
        self.n_epochs = n_epochs

    def fit(self, train_df: pd.DataFrame) -> None:
        ratings = train_df["rating"].to_numpy(dtype=np.float64)
        users = train_df["user_id"].to_numpy(dtype=np.intp)
        items = train_df["item_id"].to_numpy(dtype=np.intp)

        self.global_mean = float(ratings.mean())
        self.user_bias = np.zeros(int(users.max()) + 1, dtype=np.float64)
        self.item_bias = np.zeros(int(items.max()) + 1, dtype=np.float64)

        n_users = len(self.user_bias)
        n_items = len(self.item_bias)

        user_counts = np.bincount(users, minlength=n_users)
        item_counts = np.bincount(items, minlength=n_items)

        for _ in range(self.n_epochs):
            # Os vieses de usuário e item são atualizados alternadamente, mantendo o outro fixo.
            # cada um resolve \min_b (sum_{(u,i) in R} (r_{ui} - mu - b_u - b_i)^2 + lambda * b^2)
            # Que Ã© equivalente Ã  agrupar por usuÃ¡rio e item, e entÃ£o resolver a equaÃ§Ã£o normal para cada um.

            user_residuals = ratings - self.global_mean - self.item_bias[items]
            self.user_bias = np.bincount(
                users, weights=user_residuals, minlength=n_users
            ) / (user_counts + self.user_reg)

            item_residuals = ratings - self.global_mean - self.user_bias[users]
            self.item_bias = np.bincount(
                items, weights=item_residuals, minlength=n_items
            ) / (item_counts + self.item_reg)

    def predict(
        self,
        user_id: NDArray[np.intp],
        item_id: NDArray[np.intp],
    ) -> NDArray[np.float64]:
        return (
            self.global_mean
            + self.user_bias[user_id]
            + self.item_bias[item_id]
        )

#### Hipótese e previsão — Tarefa 1.1 (baseline aditiva)

**Hipótese.** Boa parte da variância das notas é explicada por efeitos aditivos: alguns usuários são mais generosos e alguns filmes são sistematicamente melhores. Com regularização $L_2$ adequada, os vieses de filmes pouco avaliados são encolhidos em direção a zero.

**Previsão.** (i) A baseline reduz o RMSE de forma clara em relação à média global (esperado: de ~1,12 para ~0,94–0,95). (ii) O RMSE de validação tem um mínimo em regularização intermediária: valores muito baixos sobreajustam os vieses dos filmes raros, valores muito altos apagam o sinal. (iii) Os maiores vieses positivos de filmes, antes da regularização, viriam de filmes com poucas notas; com a regularização eles são substituídos por filmes bem avaliados e populares.

**O que refutaria.** Se a média global tivesse RMSE próximo da baseline, ou se o RMSE não variasse com $\lambda$.

In [ ]:
BEST["baseline"] = None
# Referências triviais (também passam pelo mesmo protocolo)
final_eval("Média global", GlobalMean(), train_df, test_df)
final_eval("Popularidade (só listas)", PopularityModel(), train_df, test_df, rating_metrics=False)

grid = {"user_reg": [1, 3, 10, 25, 50], "item_reg": [1, 3, 10, 25, 50]}
res_base = grid_search(BaselineModel, grid, fixed={"n_epochs": 20}, verbose=False)
res_base.to_csv(RES_DIR / "cv_baseline.csv", index=False)
best = best_row(res_base)
BEST["baseline"] = {"user_reg": best["user_reg"], "item_reg": best["item_reg"]}
print("Melhor configuração (RMSE de validação):", BEST["baseline"], f"-> {best['val_rmse']:.4f} ± {best['val_rmse_std']:.4f}")

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
sns.heatmap(res_base.pivot(index="user_reg", columns="item_reg", values="val_rmse"), annot=True, fmt=".4f", cmap="viridis_r", ax=ax[0])
ax[0].set_title("RMSE de validação (média dos 5 folds)")
sns.heatmap(res_base.pivot(index="user_reg", columns="item_reg", values="val_rmse_std"), annot=True, fmt=".4f", cmap="Greys", ax=ax[1])
ax[1].set_title("Desvio padrão entre folds")
plt.tight_layout(); plt.savefig(FIG_DIR / "cv_baseline.png", dpi=150); plt.show()

baseline = BaselineModel(n_epochs=20, **BEST["baseline"])
out = final_eval("Baseline aditiva", baseline, train_df, test_df)
print({k: fmt_ci(v) if isinstance(v, tuple) else round(v, 3) for k, v in out.items() if k != "model"})

# O que os vieses aprendidos dizem?
item_n_train = np.bincount(train_df["item_id"], minlength=N_ITEMS)
bias_items = pd.DataFrame({"titulo": movies["title"], "bias": baseline.item_bias, "n_notas": item_n_train})
print("\nMaiores vieses de filme:\n", bias_items.sort_values("bias", ascending=False).head(8).round(3))
print("\nMenores vieses de filme:\n", bias_items.sort_values("bias").head(8).round(3))

bias_users = users.assign(bias=baseline.user_bias, n_notas=np.bincount(train_df["user_id"], minlength=N_USERS))
print("\nViés médio do usuário por gênero:\n", bias_users.groupby("gender")["bias"].mean().round(3))
print("Correlação viés do usuário × idade: %.3f" % bias_users[["bias", "age"]].corr().iloc[0, 1])
print("Correlação viés do filme × log(nº de notas): %.3f" % np.corrcoef(baseline.item_bias, np.log1p(item_n_train))[0, 1])

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].hist(baseline.user_bias, bins=40, color="C1"); ax[0].set_title("Viés dos usuários $b_u$")
ax[1].hist(baseline.item_bias, bins=40, color="C2"); ax[1].set_title("Viés dos filmes $b_i$")
ax[2].scatter(item_n_train, baseline.item_bias, s=6, alpha=0.5); ax[2].set_xscale("log")
ax[2].set_title("Viés do filme × nº de notas (encolhimento)"); ax[2].set_xlabel("nº de notas no treino")
plt.tight_layout(); plt.savefig(FIG_DIR / "baseline_vieses.png", dpi=150); plt.show()

**Discussão:**
*Escreva suas conclusões aqui*

---
### Introduzindo vetores latentes

A baseline linear captura diferenças sistemáticas entre usuários e itens, mas não explica por que determinados grupos de usuários atribuem notas semelhantes a determinados grupos de itens.
Para investigar essa estrutura, podemos olhar para a matriz de notas como um todo e perguntar se sua variabilidade pode ser explicada por um número reduzido de dimensões.
Essa perspectiva permite explorar interações globais nos dados e construir uma representação mais compacta das avaliações, codificando usuários e itens em um espaço latente compartilhado.

Uma maneira de investigar essa hipótese é completar a matriz de notas e aplicar Análise de Componentes Principais (PCA, *Principal Component Analysis*).
Para isso, substituímos as notas ausentes pela média do respectivo item, calculada apenas com as avaliações observadas no conjunto de treinamento.
Em seguida, o PCA identifica as principais direções de variação da matriz completada, permitindo representar cada usuário e cada item como vetores em um espaço de baixa dimensionalidade.

Seja $\tilde{R}$ a matriz de notas em que cada nota ausente foi substituída por um valor de imputação, por exemplo, a média do item.
O PCA com $k$ componentes aproxima $\tilde{R}$ por uma matriz de posto $k$:

$$\hat{R}_{ui} = \bar{R}_i + \mathbf{z}_u^\top \mathbf{w}_i$$

onde $\bar{R}_i$ é a média da coluna $i$ de $\tilde{R}$, $\mathbf{z}_u$ e $\mathbf{w}_i$ são as coordenadas do usuário $u$ e do item $i$ nos $k$ primeiros componentes principais.
Sob a perspectiva do PCA, um usuário é codificado como um vetor de pesos que indica a importância de cada componente principal para aquele usuário, que então é decodificado em uma nota para cada item, através da combinação linear com a transformação dos itens no mesmo espaço latente.

> **Tarefa 1.2**
Treine o modelo de PCA com imputação de notas ausentes, utilizando a média de cada item para preencher os valores faltantes. Utilize o protocolo experimental para selecionar o número de componentes principais $k$ e então reporte RMSE e MAE no conjunto de teste para o valor de $k$ selecionado.
>
> Analise e discuta também como o desempenho varia com o número de componentes, discutindo se o aumento da dimensionalidade melhora as previsões, produz ganhos decrescentes ou degrada o desempenho.


In [ ]:
class ImputedPCA:
    def __init__(self, n_components: int = 10) -> None:
        self.n_components = n_components
        self.user_embeddings: NDArray[np.float64]
        self.item_embeddings: NDArray[np.float64]
        self.item_means: NDArray[np.float64]
        self.pca = PCA(n_components=n_components)

    def fit(self, train_df: pd.DataFrame) -> None:
        X = ratings_to_matrix(train_df)

        # Imputação de valores ausentes com a média do item
        observed = ~np.isnan(X)
        global_mean = float(X[observed].mean())
        counts = observed.sum(axis=0)
        sums = np.where(observed, X, 0.0).sum(axis=0)

        self.item_means = np.divide(
            sums,
            counts,
            out=np.full(X.shape[1], global_mean),
            where=counts > 0,
        )

        rows, cols = np.where(~observed)
        X[rows, cols] = self.item_means[cols]

        user_scores = self.pca.fit_transform(X)
        self.user_embeddings = user_scores
        self.item_embeddings = self.pca.components_.T

    def predict(
        self,
        user_id: NDArray[np.intp],
        item_id: NDArray[np.intp],
    ) -> NDArray[np.float64]:
        return (
            self.pca.mean_[item_id]
            + np.sum(
                self.user_embeddings[user_id]
                * self.item_embeddings[item_id],
                axis=1,
            )
        )

#### Hipótese e previsão — Tarefa 1.2 (PCA com imputação)

**Hipótese.** Preencher as notas faltantes com a média do filme faz o PCA enxergar principalmente as notas observadas como desvios em relação a um fundo constante. Como mais de 93% das células são imputadas, os componentes principais são dominados pela variância dos poucos valores observados e pela estrutura de quais pares foram avaliados, não apenas pelas preferências.

**Previsão.** (i) O RMSE melhora com poucos componentes, em relação a $k=0$ (só a média do filme, que é essencialmente a baseline sem viés de usuário), e depois deixa de melhorar. (ii) Para $k$ grande, o desempenho de validação **piora**, porque com mais componentes o modelo reconstrói perfeitamente as células observadas do treino, que ele vê como desvios em relação a um fundo imputado. (iii) A variância explicada se concentra nos primeiros componentes.

**O que refutaria.** Se o RMSE de validação continuasse caindo até $k$ da ordem de centenas, a imputação não estaria distorcendo a estrutura aprendida.

In [ ]:
grid = {"n_components": [1, 2, 3, 5, 8, 10, 15, 20, 30, 50, 100]}
res_pca = grid_search(ImputedPCA, grid, verbose=False)
res_pca.to_csv(RES_DIR / "cv_pca_imputado.csv", index=False)
best = best_row(res_pca)
BEST["pca"] = {"n_components": int(best["n_components"])}
print("Melhor k:", BEST["pca"], f"-> RMSE de validação {best['val_rmse']:.4f} ± {best['val_rmse_std']:.4f}")
print(res_pca[["n_components", "val_rmse", "val_rmse_std", "val_mae", "fit_s"]].round(4).to_string(index=False))

pca_model = ImputedPCA(**BEST["pca"])
out = final_eval("PCA imputado", pca_model, train_df, test_df)
print({k: fmt_ci(v) if isinstance(v, tuple) else round(v, 3) for k, v in out.items() if k != "model"})

# Variância explicada pela imputação: ajustamos um PCA completo na matriz imputada de treino
X = ratings_to_matrix(train_df)
frac_imputed = np.isnan(X).mean()
ev = PCA().fit(np.where(np.isnan(X), np.nanmean(X, axis=0), X)).explained_variance_ratio_
print(f"\nfração de células imputadas: {frac_imputed:.1%}")
print("variância explicada acumulada: k=10 -> %.1f%%, k=50 -> %.1f%%, k=100 -> %.1f%%" % (
    100 * ev[:10].sum(), 100 * ev[:50].sum(), 100 * ev[:100].sum()))

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
plot_cv_curve(res_pca, "n_components", ax=ax[0], logx=True)
ax[0].set_title("PCA imputado: RMSE de validação × k")
ax[1].plot(np.arange(1, 101), np.cumsum(ev[:100]), marker=".")
ax[1].set_title("Variância explicada acumulada (matriz imputada)"); ax[1].set_xlabel("componentes")
plt.tight_layout(); plt.savefig(FIG_DIR / "cv_pca_imputado.png", dpi=150); plt.show()

**Discussão:**
*Escreva suas conclusões aqui*

---

### Fatoração de matrizes (*Matrix Factorization*)

A principal limitação do modelo anterior é que ele depende de notas imputadas, que não correspondem a avaliações reais e podem introduzir vieses no aprendizado dos vetores latentes.
Para superar essa limitação, devemos aprender os vetores latentes diretamente a partir das notas observadas, sem precisar completar previamente a matriz.

A fatoração de matrizes parte da hipótese de que a matriz de notas pode ser aproximada pelo produto de duas matrizes de menor dimensão, ou seja

$$\hat R = Z W^\top,$$
em que $Z\in\mathbb R^{|U|\times k}$ e $W\in\mathbb R^{|I|\times k}$ contêm os vetores latentes dos usuários e dos itens, respectivamente.
Dessa forma, a nota prevista para um par usuÃ¡rioâ€“item Ã© dada pelo produto interno entre suas representaÃ§Ãµes:

$$\hat R_{ui}=\mathbf z_u^\top\mathbf w_i.$$

Entretanto, como vimos na baseline, parte da variação das notas pode ser explicada por diferenças sistemáticas entre usuários e itens, independentemente de suas interações (por exemplo, alguns filmes são mais populares que outros).
Para considerar esses efeitos, incorporamos a média global e os *biases* de usuário e item ao modelo, obtendo a formulação geral com $f(\mathbf{z}_u, \mathbf{w}_i) = \mathbf z_u^\top\mathbf w_i$:

$$\hat R_{ui}=\mu+b_u+b_i+\mathbf z_u^\top\mathbf w_i.$$

Para aprender os parÃ¢metros, minimizamos o erro quadrÃ¡tico entre as notas observadas e as respectivas previsÃµes, considerando apenas os pares usuÃ¡rioâ€“item avaliados no conjunto de treinamento.
Introduzimos também regularização $L_2$, que penaliza valores elevados dos *biases* e das normas dos vetores latentes:

\begin{aligned}
\min_{\mu,b,Z_U,Z_I}\quad
&\sum_{(u,i)\in\Omega}
\left(r_{ui}-\mu-b_u-b_i-\mathbf z_u^\top\mathbf w_i\right)^2\\
&+\lambda_b\left(\sum_u b_u^2+\sum_i b_i^2\right)\\
&+\lambda_z\left(\sum_u\|\mathbf z_u\|_2^2+
\sum_i\|\mathbf w_i\|_2^2\right),
\end{aligned}

em que $\Omega$ é o conjunto de avaliações observadas no treinamento, enquanto $\lambda_b$ e $\lambda_z$ controlam a intensidade da regularização dos *biases* e dos fatores latentes, respectivamente.
Como o problema é não-convexo, sem solução fechada, usamos descida no gradiente para atualizar os parâmetros, iterativamente com base em pequenas amostras do conjunto de treinamento.

> **Tarefa 1.3**
Treine um modelo de fatoração de matrizes utilizando regularização $L_2$.
Selecione o número de dimensões latentes $k$ e os coeficientes de regularização com base no RMSE.
> 
> Reporte RMSE e MAE no conjunto de teste para a configuração selecionada.
Analise como o desempenho varia com $k$ e com a regularização.
Por fim, compare a fatoração com *biases* e sem *biases*, mantendo o mesmo protocolo experimental, para investigar a contribuição dos efeitos aditivos e das interações latentes para a qualidade das previsões.
>
> Siga o mesmo protocolo experimental utilizado para os modelos anteriores.


In [ ]:
class MatrixFactorization:
    user_embeddings: NDArray[np.float64]
    item_embeddings: NDArray[np.float64]
    global_mean: float
    user_bias: NDArray[np.float64]
    item_bias: NDArray[np.float64]

    def __init__(
        self,
        n_factors: int = 10,
        n_epochs: int = 20,
        learning_rate: float = 0.01,
        reg: float = 0.1,
        bias_reg: float | None = None,
        use_biases: bool = True,
    ) -> None:
        self.n_factors = n_factors
        self.n_epochs = n_epochs
        self.learning_rate = learning_rate
        self.reg = reg
        self.bias_reg = reg if bias_reg is None else bias_reg
        self.use_biases = use_biases

    def fit(self, train_df: pd.DataFrame) -> None:
        ratings = train_df["rating"].to_numpy(dtype=np.float64)
        users = train_df["user_id"].to_numpy(dtype=np.intp)
        items = train_df["item_id"].to_numpy(dtype=np.intp)

        self.global_mean = float(ratings.mean())
        n_users = int(users.max()) + 1
        n_items = int(items.max()) + 1

        self.user_embeddings = np.random.normal(scale=0.1, size=(n_users, self.n_factors))
        self.item_embeddings = np.random.normal(scale=0.1, size=(n_items, self.n_factors))
        self.user_bias = np.zeros(n_users, dtype=np.float64)
        self.item_bias = np.zeros(n_items, dtype=np.float64)

        for _ in range(self.n_epochs):
            for n in np.random.permutation(len(ratings)):
                u, i, r = users[n], items[n], ratings[n]
                pred = (
                    self.global_mean
                    + self.user_bias[u]
                    + self.item_bias[i]
                    + np.dot(self.user_embeddings[u], self.item_embeddings[i])
                )
                err = r - pred

                # Atualização dos vieses
                if self.use_biases:
                    self.user_bias[u] += self.learning_rate * (err - self.bias_reg * self.user_bias[u])
                    self.item_bias[i] += self.learning_rate * (err - self.bias_reg * self.item_bias[i])

                user_emb_old = self.user_embeddings[u].copy()
                item_emb_old = self.item_embeddings[i].copy()

                self.user_embeddings[u] += self.learning_rate * (err * item_emb_old - self.reg * user_emb_old)
                self.item_embeddings[i] += self.learning_rate * (err * user_emb_old - self.reg * item_emb_old)

    def predict(
        self,
        user_id: NDArray[np.intp],
        item_id: NDArray[np.intp],
    ) -> NDArray[np.float64]:
        return (
            self.global_mean
            + self.user_bias[user_id]
            + self.item_bias[item_id]
            + np.sum(
                self.user_embeddings[user_id] * self.item_embeddings[item_id],
                axis=1,
            )
        )


In [ ]:
# Versão acelerada da fatoração de matrizes (mesmo modelo, mesmas atualizações, mesma ordem de sorteios).
# A implementação de referência percorre as notas em um laço Python; aqui o laço do SGD é compilado com numba.
# Cada passo do SGD é idêntico ao da classe `MatrixFactorization` e o gerador aleatório é consumido na mesma ordem,
# então, com a mesma semente, os parâmetros resultantes coincidem (verificado abaixo).
from numba import njit


@njit
def _sgd_epoch(order, users, items, ratings, global_mean, user_bias, item_bias, U, V, lr, reg, bias_reg, use_biases):
    k = U.shape[1]
    for idx in range(order.shape[0]):
        n = order[idx]
        u, i, r = users[n], items[n], ratings[n]
        dot = 0.0
        for f in range(k):
            dot += U[u, f] * V[i, f]
        err = r - (global_mean + user_bias[u] + item_bias[i] + dot)
        if use_biases:
            user_bias[u] += lr * (err - bias_reg * user_bias[u])
            item_bias[i] += lr * (err - bias_reg * item_bias[i])
        for f in range(k):
            u_old, v_old = U[u, f], V[i, f]
            U[u, f] += lr * (err * v_old - reg * u_old)
            V[i, f] += lr * (err * u_old - reg * v_old)


class FastMatrixFactorization(MatrixFactorization):
    """Fatoração de matrizes com SGD compilado (numba). Aceita `track_df` para registrar o RMSE por época."""

    def fit(self, train_df: pd.DataFrame, track_df: pd.DataFrame | None = None) -> None:
        ratings = train_df["rating"].to_numpy(dtype=np.float64)
        users = train_df["user_id"].to_numpy(dtype=np.int64)
        items = train_df["item_id"].to_numpy(dtype=np.int64)

        self.global_mean = float(ratings.mean())
        n_users, n_items = int(users.max()) + 1, int(items.max()) + 1
        self.user_embeddings = np.random.normal(scale=0.1, size=(n_users, self.n_factors))
        self.item_embeddings = np.random.normal(scale=0.1, size=(n_items, self.n_factors))
        self.user_bias = np.zeros(n_users)
        self.item_bias = np.zeros(n_items)
        self.history: list[float] = []

        for _ in range(self.n_epochs):
            order = np.random.permutation(len(ratings)).astype(np.int64)
            _sgd_epoch(order, users, items, ratings, self.global_mean, self.user_bias, self.item_bias,
                       self.user_embeddings, self.item_embeddings, self.learning_rate, self.reg,
                       self.bias_reg, self.use_biases)
            if track_df is not None:
                pred = clip_ratings(self.predict(track_df["user_id"].to_numpy(), track_df["item_id"].to_numpy()))
                self.history.append(root_mean_squared_error(track_df["rating"].to_numpy(), pred))


# Verificação de fidelidade: mesma semente => mesmos parâmetros que a implementação de referência
_sub = train_df.sample(3000, random_state=0)
set_seed(7); _ref = MatrixFactorization(n_factors=4, n_epochs=2); _ref.fit(_sub)
set_seed(7); _fast = FastMatrixFactorization(n_factors=4, n_epochs=2); _fast.fit(_sub)
assert np.allclose(_ref.user_embeddings, _fast.user_embeddings) and np.allclose(_ref.item_embeddings, _fast.item_embeddings)
assert np.allclose(_ref.user_bias, _fast.user_bias) and np.allclose(_ref.item_bias, _fast.item_bias)
print("FastMatrixFactorization reproduz a implementação de referência (max |diff| = %.2e)" %
      np.abs(_ref.user_embeddings - _fast.user_embeddings).max())

#### Hipótese e previsão — Tarefa 1.3 (fatoração de matrizes, com e sem vieses)

**Hipótese.** Aprender os vetores latentes só com as notas observadas elimina a distorção da imputação. Mas os efeitos aditivos (viés de usuário e de filme) já explicam uma parte grande da variância; sem eles, o produto interno $\mathbf z_u^\top\mathbf w_i$ precisa reaprender "este filme é bom" e "este usuário é generoso" usando dimensões latentes.

**Previsão.** (i) A fatoração **com** vieses supera a baseline aditiva, com ganho pequeno (da ordem de 0,01–0,03 de RMSE), porque as interações latentes acrescentam pouco sobre os efeitos aditivos num conjunto tão esparso. (ii) A fatoração **sem** vieses é pior que a com vieses, e sua escolha de $k$ e de $\lambda_z$ é mais sensível. (iii) O RMSE de validação em função de $k$ forma uma curva em U, e a melhor regularização cresce com $k$. (iv) **Listas:** o modelo de menor RMSE não será necessariamente o de melhores listas; a popularidade, sozinha, deve ser uma referência forte em Precision@10, e os modelos que só ajustam o erro quadrático de notas podem ficar abaixo dela.

**O que refutaria.** (i) Se a fatoração com vieses não superasse a baseline em RMSE; (ii) se a versão sem vieses empatasse com a com vieses; (iv) se o ranking de modelos por RMSE fosse idêntico ao ranking por NDCG@10 e todos superassem a popularidade.

In [ ]:
MF_EPOCHS = 30

# --- Com vieses ---
grid_b = {"n_factors": [5, 10, 20, 40], "reg": [0.03, 0.06, 0.1], "bias_reg": [0.03, 0.1]}
res_mf_b = grid_search(FastMatrixFactorization, grid_b, fixed={"n_epochs": MF_EPOCHS, "use_biases": True}, verbose=False)
res_mf_b.to_csv(RES_DIR / "cv_mf_com_vieses.csv", index=False)
best_b = best_row(res_mf_b)
BEST["mf_bias"] = {"n_factors": int(best_b["n_factors"]), "reg": float(best_b["reg"]), "bias_reg": float(best_b["bias_reg"])}
print("MF com vieses:", BEST["mf_bias"], f"-> {best_b['val_rmse']:.4f} ± {best_b['val_rmse_std']:.4f}")

# --- Sem vieses (bias_reg é irrelevante; mesma grade para k e reg) ---
grid_nb = {"n_factors": [5, 10, 20, 40], "reg": [0.03, 0.06, 0.1]}
res_mf_nb = grid_search(FastMatrixFactorization, grid_nb, fixed={"n_epochs": MF_EPOCHS, "use_biases": False}, verbose=False)
res_mf_nb.to_csv(RES_DIR / "cv_mf_sem_vieses.csv", index=False)
best_nb = best_row(res_mf_nb)
BEST["mf_nobias"] = {"n_factors": int(best_nb["n_factors"]), "reg": float(best_nb["reg"])}
print("MF sem vieses:", BEST["mf_nobias"], f"-> {best_nb['val_rmse']:.4f} ± {best_nb['val_rmse_std']:.4f}")

# --- Convergência por época na validação (fold 0) das configurações escolhidas ---
tr0, va0 = FOLDS[0]
curves = {}
for name, cfg, use_b in [("com vieses", BEST["mf_bias"], True), ("sem vieses", BEST["mf_nobias"], False)]:
    set_seed(SEED)
    m = FastMatrixFactorization(n_epochs=80, use_biases=use_b, **cfg)
    m.fit(tr0, track_df=va0)
    curves[name] = m.history
    print(f"{name}: melhor época = {int(np.argmin(m.history)) + 1}, RMSE = {min(m.history):.4f}")

fig, ax = plt.subplots(1, 3, figsize=(18, 4.5))
for reg, grp in res_mf_b[res_mf_b["bias_reg"] == BEST["mf_bias"]["bias_reg"]].groupby("reg"):
    ax[0].errorbar(grp["n_factors"], grp["val_rmse"], yerr=grp["val_rmse_std"], marker="o", capsize=3, label=f"λz={reg}")
ax[0].set_title("MF com vieses: RMSE de validação × k"); ax[0].set_xlabel("k"); ax[0].legend(fontsize=8)
for reg, grp in res_mf_nb.groupby("reg"):
    ax[1].errorbar(grp["n_factors"], grp["val_rmse"], yerr=grp["val_rmse_std"], marker="o", capsize=3, label=f"λz={reg}")
ax[1].set_title("MF sem vieses: RMSE de validação × k"); ax[1].set_xlabel("k"); ax[1].legend(fontsize=8)
for name, hist in curves.items():
    ax[2].plot(np.arange(1, len(hist) + 1), hist, label=name)
ax[2].axvline(MF_EPOCHS, ls="--", c="gray"); ax[2].set_title("RMSE de validação × época (fold 0)")
ax[2].set_xlabel("época"); ax[2].legend()
plt.tight_layout(); plt.savefig(FIG_DIR / "cv_mf.png", dpi=150); plt.show()

# --- Avaliação final no teste ---
for name, cfg, use_b in [("MF sem vieses", BEST["mf_nobias"], False), ("MF com vieses", BEST["mf_bias"], True)]:
    out = final_eval(name, FastMatrixFactorization(n_epochs=MF_EPOCHS, use_biases=use_b, **cfg), train_df, test_df)
    print(name, {k: fmt_ci(v) for k, v in out.items() if isinstance(v, tuple)})

print("\nTabela parcial (teste):")
display(final_table())

**Discussão:**
*Escreva suas conclusões aqui*

---
## Etapa 2: Suposições sobre a estrutura latente

Na etapa anterior, investigamos modelos que estimam notas a partir de vieses aditivos e representações latentes de usuários e itens.
A fatoração de matrizes procura representações que aproximem as avaliações observadas, mas não define, por si só, uma distribuição probabilística completa para os dados.

Nesta etapa, exploraremos uma abordagem probabilística baseada no PCA Probabilístico (*Probabilistic Principal Component Analysis*, PPCA), adaptada ao problema de recomendação.
Essa formulação descreve as avaliações por meio de variáveis latentes e ruído gaussiano, permitindo explicitar a variância do ruído, estimar a verossimilhança dos dados e investigar como as suposições probabilísticas afetam as previsões.

Ao contrário das implementações clássicas de PPCA e VAE, em que o encoder é treinado para mapear as observações para a distribuição posterior, aqui, como os usuários do conjunto de dados são conhecidos durante o treinamento, podemos estimar um embedding para cada um deles e armazenar essas representações após o ajuste do modelo.
Isso é equivalente a um *encoder* que mapeia cada usuário para uma distribuição posterior com o máximo de liberdade, sem lidar com features de usuário, vieses indutivos ou generalização para novos usuários.


### PCA Probabilístico

O PCA clássico identifica um subespaço linear de dimensão reduzida que representa a estrutura de variância dos dados. O PPCA estende essa ideia ao assumir que as observações são geradas por transformações lineares de variáveis latentes gaussianas, acrescidas de ruído gaussiano.

No problema de recomendação, podemos representar a matriz de avaliações por $R\in\mathbb R^{n\times m}$, em que $n$ é o número de usuários e $m$ é o número de filmes. Cada usuário corresponde a uma observação, e cada filme corresponde a uma dimensão observada. Como os usuários do conjunto de dados são conhecidos durante o treinamento, podemos estimar um embedding para cada um deles e armazenar essas representações após o ajuste do modelo.

Seja $\mathbf z_u\in\mathbb R^k$ a variável latente do usuário $u$, e seja $\mathbf w_i\in\mathbb R^k$ o vetor de cargas associado ao filme $i$. O modelo assume que

$$
\mathbf z_u\sim\mathcal N(\mathbf 0,I_k)
$$

e que a avaliação do usuário $u$ para o filme $i$ segue

$$
R_{ui}\mid\mathbf z_u
\sim
\mathcal N\left(
\mu_i+\mathbf z_u^\top\mathbf w_i,
\sigma^2
\right),
$$

em que $\mu_i$ é a média associada ao filme $i$, $k$ é a dimensão latente e $\sigma^2$ é a variância do ruído, considerada fixa durante o treinamento.
A média condicional da avaliação é, portanto,

$$
\mathbb E[R_{ui}\mid\mathbf z_u]
=
\mu_i+\mathbf z_u^\top\mathbf w_i.
$$

Essa formulação representa as avaliações por meio de uma estrutura linear de baixa dimensão, sem introduzir vieses aditivos específicos de usuários.
A diferenÃ§a fundamental em relaÃ§Ã£o Ã  fatoraÃ§Ã£o de matrizes determinÃ­stica estÃ¡ na distribuiÃ§Ã£o probabilÃ­stica atribuÃ­da Ã s avaliaÃ§Ãµes e Ã s variÃ¡veis latentes.

O treinamento pode ser realizado pelo algoritmo EM (*Expectation-Maximization*).
Seja $\Omega_u$ o conjunto de filmes avaliados pelo usuário $u$ e $\Omega_i$ o conjunto de usuários que avaliaram o filme $i$.
Na etapa E, como o modelo Ã© linear e gaussiano, a posterior de cada usuÃ¡rio, condicionada Ã s avaliaÃ§Ãµes observadas e aos parÃ¢metros atuais, Ã© gaussiana, $p(\mathbf z_u\mid R_u)=\mathcal N(\mathbf m_u, S_u)$, com

$$
S_u=\left( I_k + \frac{1}{\sigma^2} \sum_{i\in\Omega_u} \mathbf w_i\mathbf w_i^\top\right)^{-1},
\qquad
\mathbf m_u=\frac{1}{\sigma^2} \, S_u\sum_{i\in\Omega_u}\mathbf w_i\,(r_{ui}-\mu_i).
$$

Na etapa M, atualizamos as médias dos filmes e os vetores de cargas para maximizar a verossimilhança esperada.
Definindo $\boldsymbol\theta_i=(\mu_i,\mathbf w_i)$ e $\boldsymbol\phi_u=(1,\mathbf z_u)$, cada filme resolve um sistema de mínimos quadrados em que as posteriores entram pelos momentos $\mathbb E[\boldsymbol\phi_u]=(1,\mathbf m_u)$ e $\mathbb E[\mathbf z_u\mathbf z_u^\top]=S_u+\mathbf m_u\mathbf m_u^\top$:

$$
\boldsymbol\theta_i=\left(\sum_{u\in\Omega_i}\mathbb E[\boldsymbol\phi_u\boldsymbol\phi_u^\top]\right)^{-1}\sum_{u\in\Omega_i}r_{ui}\,\mathbb E[\boldsymbol\phi_u].
$$

Como a matriz de avaliaÃ§Ãµes Ã© esparsa, cada usuÃ¡rio contribui para a estimaÃ§Ã£o apenas por meio dos filmes que avaliou. As avaliaÃ§Ãµes ausentes sÃ£o tratadas como dados nÃ£o observados, e nÃ£o como notas iguais a zero. ApÃ³s o treinamento, as mÃ©dias dos embeddings posteriores podem ser armazenadas como representaÃ§Ãµes dos usuÃ¡rios, permitindo realizar previsÃµes diretamente para os pares usuÃ¡rioâ€“filme.

Para uma avaliação observada $r_{ui}$, a densidade condicional é

$$
p(r_{ui}\mid\mathbf z_u)
=
\frac{1}{\sqrt{2\pi\sigma^2}}
\exp\left(
-\frac{
(r_{ui}-\mu_i-\mathbf w_i^\top\mathbf z_u)^2
}{2\sigma^2}
\right).
$$

A log-verossimilhanÃ§a conjunta das avaliaÃ§Ãµes observadas, condicionada Ã s variÃ¡veis latentes, Ã© obtida somando as contribuiÃ§Ãµes dos pares usuÃ¡rioâ€“filme observados. O prior gaussiano das variÃ¡veis latentes tambÃ©m participa do modelo probabilÃ­stico, favorecendo representaÃ§Ãµes compatÃ­veis com a distribuiÃ§Ã£o $\mathcal N(\mathbf 0,I_k)$.

Para prever a avaliaÃ§Ã£o de um par usuÃ¡rioâ€“filme, utilizamos a mÃ©dia condicional calculada com o embedding posterior estimado:

$$
\hat R_{ui}
=
\mu_i+\mathbf w_i^\top\mathbb E[\mathbf z_u\mid R_u],
$$

em que $R_u$ representa as avaliaÃ§Ãµes observadas do usuÃ¡rio $u$. Essa previsÃ£o corresponde Ã  mÃ©dia da distribuiÃ§Ã£o preditiva quando os parÃ¢metros do modelo sÃ£o considerados fixos e a incerteza sobre a variÃ¡vel latente Ã© integrada.

A variÃ¢ncia $\sigma^2$ controla o ruÃ­do residual assumido pelo modelo. Valores menores atribuem maior concentraÃ§Ã£o Ã s avaliaÃ§Ãµes em torno da mÃ©dia condicional; valores maiores representam maior dispersÃ£o. Como a variÃ¢ncia Ã© fixada durante o treinamento nesta implementaÃ§Ã£o, ela deve ser selecionada como hiperparÃ¢metro.

> **Tarefa 2.1**
Treine o modelo PPCA para a matriz esparsa de avaliações e selecione o número de dimensões latentes $k$ e a variância do ruído $\sigma^2$ usando o protocolo experimental.
Compare o RMSE com os modelos da etapa anterior e investigue se a configuração que minimiza o RMSE também produz a melhor log-verossimilhança negativa média (*Negative Log-Likelihood*, NLL).
>
> Discuta como a variância do ruído afeta a distribuição das avaliações, como o prior gaussiano influencia as representações latentes e quais são as implicações de estimar embeddings específicos para os usuários conhecidos durante o treinamento.

In [ ]:
class PPCA:
    user_embeddings: NDArray[np.float64]
    item_embeddings: NDArray[np.float64]
    item_means: NDArray[np.float64]
    user_covariances: NDArray[np.float64]

    def __init__(
        self, n_factors: int = 10, n_epochs: int = 100, sigma2: float = 1.0
    ) -> None:
        self.n_factors = n_factors
        self.n_epochs = n_epochs
        self.sigma2 = sigma2

    def fit(self, train_df: pd.DataFrame) -> None:
        X = ratings_to_matrix(train_df)
        observed = ~np.isnan(X)
        values = np.where(observed, X, 0.0)
        weights = observed.astype(np.float64)

        counts = observed.sum(axis=0)
        global_mean = values.sum() / observed.sum()
        self.item_means = np.divide(
            values.sum(axis=0),
            counts,
            out=np.full(X.shape[1], global_mean),
            where=counts > 0,
        )

        n_users, n_items = X.shape
        k = self.n_factors
        sigma2 = self.sigma2
        eye = np.eye(k)

        self.item_embeddings = np.random.normal(
            scale=0.1, size=(n_items, k)
        )

        item_outer = (
            self.item_embeddings[:, :, None]
            * self.item_embeddings[:, None, :]
        )

        for _ in range(self.n_epochs):
            centered = np.where(observed, X - self.item_means, 0.0)

            precision = eye + np.einsum("ui,ijk->ujk", weights, item_outer) / sigma2

            self.user_covariances = np.linalg.inv(precision)

            rhs = centered @ self.item_embeddings / sigma2
            self.user_embeddings = (
                self.user_covariances @ rhs[:, :, None]
            )[:, :, 0]

            user_means = np.column_stack(
                (np.ones(n_users), self.user_embeddings)
            )

            user_second = np.zeros((n_users, k + 1, k + 1))
            user_second[:, 0, 0] = 1.0
            user_second[:, 0, 1:] = self.user_embeddings
            user_second[:, 1:, 0] = self.user_embeddings
            user_second[:, 1:, 1:] = (
                self.user_covariances
                + self.user_embeddings[:, :, None]
                * self.user_embeddings[:, None, :]
            )

            A = np.einsum("ui,uab->iab", weights, user_second)

            b = values.T @ user_means
            theta = np.linalg.solve(A, b[:, :, None])[:, :, 0]

            self.item_means = theta[:, 0]
            self.item_embeddings = theta[:, 1:]

            item_outer = (
                self.item_embeddings[:, :, None]
                * self.item_embeddings[:, None, :]
            )

        centered = np.where(observed, X - self.item_means, 0.0)

        precision = eye + np.einsum("ui,ijk->ujk", weights, item_outer) / sigma2

        self.user_covariances = np.linalg.inv(precision)

        rhs = centered @ self.item_embeddings / sigma2
        self.user_embeddings = (
            self.user_covariances @ rhs[:, :, None]
        )[:, :, 0]

    def predict(
        self,
        user_id: NDArray[np.intp],
        item_id: NDArray[np.intp],
    ) -> NDArray[np.float64]:
        return self.item_means[item_id] + np.sum(
            self.user_embeddings[user_id] * self.item_embeddings[item_id],
            axis=1,
        )

    def predict_log_proba(
        self,
        user_id: NDArray[np.intp],
        item_id: NDArray[np.intp],
        ratings: NDArray[np.float64],
    ) -> NDArray[np.float64]:
        user_embeddings = self.user_embeddings[user_id]
        item_embeddings = self.item_embeddings[item_id]
        covariances = self.user_covariances[user_id]

        means = self.item_means[item_id] + np.sum(
            user_embeddings * item_embeddings, axis=1
        )

        variances = self.sigma2 + np.einsum(
            "ni,nij,nj->n",
            item_embeddings,
            covariances,
            item_embeddings,
        )

        return -0.5 * (
            np.log(2 * np.pi * variances)
            + (ratings - means) ** 2 / variances
        )

In [ ]:
# Versão acelerada do PPCA: as somas sobre filmes/usuários passam a ser produtos de matrizes (BLAS).
# As equações do passo E e do passo M são exatamente as da classe `PPCA`; só muda a forma de calcular
# `einsum("ui,ijk->ujk")` e `einsum("ui,uab->iab")`, que viram `weights @ outer` e `weights.T @ second`.
class FastPPCA(PPCA):
    def _e_step(self, X, observed, weights, item_outer):
        n_users, k = X.shape[0], self.n_factors
        precision = np.eye(k) + (weights @ item_outer.reshape(item_outer.shape[0], -1)).reshape(n_users, k, k) / self.sigma2
        self.user_covariances = np.linalg.inv(precision)
        centered = np.where(observed, X - self.item_means, 0.0)
        rhs = centered @ self.item_embeddings / self.sigma2
        self.user_embeddings = (self.user_covariances @ rhs[:, :, None])[:, :, 0]

    def fit(self, train_df: pd.DataFrame) -> None:
        X = ratings_to_matrix(train_df)
        observed = ~np.isnan(X)
        values = np.where(observed, X, 0.0)
        weights = observed.astype(np.float64)

        counts = observed.sum(axis=0)
        global_mean = values.sum() / observed.sum()
        self.item_means = np.divide(values.sum(axis=0), counts, out=np.full(X.shape[1], global_mean), where=counts > 0)

        n_users, n_items = X.shape
        k = self.n_factors
        self.item_embeddings = np.random.normal(scale=0.1, size=(n_items, k))
        outer = lambda W: W[:, :, None] * W[:, None, :]

        for _ in range(self.n_epochs):
            self._e_step(X, observed, weights, outer(self.item_embeddings))
            user_means = np.column_stack((np.ones(n_users), self.user_embeddings))
            second = np.zeros((n_users, k + 1, k + 1))
            second[:, 0, 0] = 1.0
            second[:, 0, 1:] = self.user_embeddings
            second[:, 1:, 0] = self.user_embeddings
            second[:, 1:, 1:] = self.user_covariances + outer(self.user_embeddings)

            A = (weights.T @ second.reshape(n_users, -1)).reshape(n_items, k + 1, k + 1)
            b = values.T @ user_means
            theta = np.linalg.solve(A, b[:, :, None])[:, :, 0]
            self.item_means, self.item_embeddings = theta[:, 0], theta[:, 1:]

        self._e_step(X, observed, weights, outer(self.item_embeddings))


# Verificação de fidelidade em relação à implementação de referência
set_seed(7); _ref = PPCA(n_factors=3, n_epochs=3, sigma2=1.0); _ref.fit(train_df)
set_seed(7); _fast = FastPPCA(n_factors=3, n_epochs=3, sigma2=1.0); _fast.fit(train_df)
assert np.allclose(_ref.item_embeddings, _fast.item_embeddings) and np.allclose(_ref.user_embeddings, _fast.user_embeddings)
assert np.allclose(_ref.item_means, _fast.item_means) and np.allclose(_ref.user_covariances, _fast.user_covariances)
print("FastPPCA reproduz a implementação de referência (max |diff| = %.2e)" % np.abs(_ref.user_embeddings - _fast.user_embeddings).max())

#### Hipótese e previsão — Tarefa 2.1 (PPCA com EM em matriz esparsa)

**Hipótese.** O PPCA tem a mesma estrutura linear da fatoração (média por filme + $\mathbf z_u^\top\mathbf w_i$), mas o prior $\mathbf z_u\sim\mathcal N(\mathbf 0,I_k)$ e o ruído fixo $\sigma^2$ fazem a regularização depender de quanto cada usuário avaliou: a posterior de um usuário com poucas notas é encolhida para a origem, e a de um usuário com muitas notas se aproxima da estimativa de mínimos quadrados. Como não há vieses de usuário, o modelo precisa absorver a generosidade do usuário no próprio espaço latente.

**Previsão.** (i) O RMSE do PPCA fica acima do da fatoração com vieses e pode ficar acima até da baseline aditiva, pela falta de $b_u$. (ii) Para $\sigma^2$ pequeno o modelo sobreajusta os usuários com poucas notas e o RMSE de validação piora; $\sigma^2$ grande encolhe tudo em direção à média do filme. (iii) A configuração que minimiza o RMSE **não** é a de menor NLL: o NLL premia calibrar a variância preditiva (que inclui a incerteza da posterior), então deve preferir $\sigma^2$ próximo da variância real dos resíduos (~0,8–0,9), enquanto o RMSE só depende da média.

**O que refutaria.** Se RMSE e NLL escolhessem a mesma configuração, ou se o PPCA superasse a fatoração com vieses.

In [ ]:
PPCA_EPOCHS = 60

# Convergência do EM: o RMSE/NLL de validação deixa de mudar com o nº de passos?
tr0, va0 = FOLDS[0]
conv = []
for n_ep in [5, 10, 20, 40, 60, 100]:
    set_seed(SEED)
    m = FastPPCA(n_factors=10, sigma2=1.0, n_epochs=n_ep)
    m.fit(tr0)
    conv.append({"passos EM": n_ep, **evaluate_model(m, val=va0)})
conv = pd.DataFrame(conv)
print(conv.round(4).to_string(index=False))

grid = {"n_factors": [2, 5, 10, 20], "sigma2": [0.4, 0.8, 1.2, 2.0]}
res_ppca = grid_search(FastPPCA, grid, fixed={"n_epochs": PPCA_EPOCHS}, verbose=False)
res_ppca.to_csv(RES_DIR / "cv_ppca.csv", index=False)

best_rmse, best_nll = best_row(res_ppca, "val_rmse"), best_row(res_ppca, "val_nll")
BEST["ppca_rmse"] = {"n_factors": int(best_rmse["n_factors"]), "sigma2": float(best_rmse["sigma2"])}
BEST["ppca_nll"] = {"n_factors": int(best_nll["n_factors"]), "sigma2": float(best_nll["sigma2"])}
print("\nMenor RMSE:", BEST["ppca_rmse"], f"RMSE={best_rmse['val_rmse']:.4f} NLL={best_rmse['val_nll']:.4f}")
print("Menor NLL :", BEST["ppca_nll"], f"RMSE={best_nll['val_rmse']:.4f} NLL={best_nll['val_nll']:.4f}")
print(res_ppca[["n_factors", "sigma2", "val_rmse", "val_rmse_std", "val_nll", "val_nll_std", "fit_s"]].round(4).to_string(index=False))

fig, ax = plt.subplots(1, 3, figsize=(18, 4.5))
sns.heatmap(res_ppca.pivot(index="n_factors", columns="sigma2", values="val_rmse"), annot=True, fmt=".4f", cmap="viridis_r", ax=ax[0])
ax[0].set_title("PPCA: RMSE de validação")
sns.heatmap(res_ppca.pivot(index="n_factors", columns="sigma2", values="val_nll"), annot=True, fmt=".3f", cmap="viridis_r", ax=ax[1])
ax[1].set_title("PPCA: NLL de validação")
ax[2].plot(conv["passos EM"], conv["val_rmse"], marker="o"); ax[2].set_xlabel("passos EM"); ax[2].set_ylabel("RMSE de validação")
ax[2].set_title("Convergência do EM (k=10, σ²=1, fold 0)")
plt.tight_layout(); plt.savefig(FIG_DIR / "cv_ppca.png", dpi=150); plt.show()

# Efeito de σ² na distribuição preditiva: largura média do intervalo de 95% e cobertura no teste
for name, cfg in [("PPCA (menor RMSE)", BEST["ppca_rmse"]), ("PPCA (menor NLL)", BEST["ppca_nll"])]:
    model = FastPPCA(n_epochs=PPCA_EPOCHS, **cfg)
    out = final_eval(name, model, train_df, test_df)
    print(name, cfg, {k: fmt_ci(v) for k, v in out.items() if isinstance(v, tuple) and k in ("rmse", "mae", "nll")})
    u, i = test_df["user_id"].to_numpy(), test_df["item_id"].to_numpy()
    var = model.sigma2 + np.einsum("ni,nij,nj->n", model.item_embeddings[i], model.user_covariances[u], model.item_embeddings[i])
    half = 1.96 * np.sqrt(var)
    mean = model.predict(u, i)
    cover = np.mean(np.abs(test_df["rating"].to_numpy() - mean) <= half)
    print(f"   desvio preditivo médio={np.sqrt(var).mean():.3f} | cobertura do intervalo de 95% no teste={cover:.1%}")

print("\nTabela parcial (teste):")
display(final_table())

**Discussão:**
*Escreva suas conclusões aqui*

### Autoencoder variacional (VAE)

O PPCA modela as avaliações por meio de uma transformação linear de variáveis latentes gaussianas. O VAE estende essa abordagem ao permitir um decodificador não linear e aprender uma distribuição latente específica para cada usuário.

Seja $k$ a dimensão latente.
O codificador é representado por uma matriz de embeddings de usuários, que fornece a média latente $\mathbf m_u$ e uma matriz de parâmetros que fornece a variância diagonal $\mathbf s_u^2$ de cada usuário.
Assim, a distribuição latente aproximada é

$$
\mathbf z_u
\sim
\mathcal N\left(
\mathbf m_u,
\operatorname{diag}(\mathbf s_u^2)
\right),
$$
que pode ser amostrada por meio do truque de reparametrização:

$$
\mathbf z_u
=
\mathbf m_u+\mathbf s_u\odot\boldsymbol\epsilon,
\qquad
\boldsymbol\epsilon\sim\mathcal N(\mathbf 0,I_k).
$$

O decodificador transforma essa amostra em previsões para todos os filmes através de uma função $g_\theta:\mathbb R^k\to\mathbb R^m$ parametrizada por $\theta$:

$$
\hat{\mathbf R}_u = g_\theta(\mathbf z_u).
$$

Assumimos o seguinte modelo generativo:

$$
R_{ui}\mid\mathbf z_u
\sim
\mathcal N\left(g_\theta(\mathbf z_u)_i,\sigma^2\right),
$$

em que $\sigma^2$ é a variância do ruído das avaliações.
Com um decodificador linear, $g_\theta(\mathbf z)=\boldsymbol\mu+W\mathbf z$, a média condicional das avaliações é uma função linear da variável latente, como no PPCA.
Um decodificador não linear permite representar relações mais complexas entre as dimensões latentes e as avaliações.

O treinamento maximiza o limite inferior da log-verossimilhança (*Evidence Lower Bound*, ELBO), somado sobre os usuários:

$$
\mathcal L_u
=
\mathbb E_{q_\phi(\mathbf z_u)}
\left[
-\frac{1}{2\sigma^2}
\sum_{i\in\Omega_u}
(r_{ui}-g_\theta(\mathbf z_u)_i)^2
\right] -\beta\,\mathrm{KL}\left(
q_\phi(\mathbf z_u)
\,\|\,\mathcal N(\mathbf 0,I_k)
\right),
$$
em que $\Omega_u$ é o conjunto de filmes avaliados pelo usuário.
Foram omitidas constantes que não dependem dos parâmetros do modelo.
Com $\beta=1$, obtemos o ELBO padrão, a menos dessas constantes, variar $\beta$ altera o peso relativo da regularização latente.

O primeiro termo incentiva o modelo a atribuir alta probabilidade Ã s notas observadas, o segundo aproxima a distribuiÃ§Ã£o latente de cada usuÃ¡rio do prior gaussiano padrÃ£o, regularizando as representaÃ§Ãµes aprendidas.
A esperança é estimada por amostragem da distribuição latente, permitindo propagar gradientes pelos parâmetros do codificador e do decodificador.

Na predição, podemos usar a média latente $\mathbf m_u$ como embedding do usuário e calcular

$$
\hat R_{ui} = g_\theta(\mathbf m_u)_i.
$$

Essa escolha fornece uma previsão pontual determinística. A distribuição preditiva completa, por sua vez, também considera a variabilidade latente e o ruído das avaliações.

Diferentemente do PPCA, que calcula a posterior de cada usuário analiticamente a partir das avaliações observadas, o VAE aprende por otimização uma distribuição latente específica para cada usuário, representada por uma média e uma variância. Essa formulação é transdutiva: as representações são aprendidas para os usuários presentes no treinamento e não se estendem automaticamente a usuários novos.

> **Tarefa 2.2**
Treine um VAE com decodificador linear e compare seu desempenho com o PPCA usando o mesmo número de dimensões latentes $k$ e a mesma variância do ruído $\sigma^2$. Compare RMSE, MAE e NLL no conjunto de teste.
>
> Em seguida, substitua o decodificador linear por um não linear e investigue se a flexibilidade adicional melhora as previsões. Escolha um hiperparâmetros para variar isoladamente, $\beta$, $\sigma^2$ ou $k$, e aplique o protocolo experimental.
Verifique a convergência pela curva de treinamento armazenada em `history` antes de comparar os modelos.

Atenção, com um decoder não-linear, os itens 

In [ ]:
import torch
from torch import nn


class VAE:
    user_embeddings: NDArray[np.float64]
    user_variances: NDArray[np.float64]
    item_embeddings: NDArray[np.float64]
    item_means: NDArray[np.float64]

    def __init__(
        self,
        n_factors: int = 10,
        n_epochs: int = 300,
        sigma2: float = 1.0,
        beta: float = 1.0,
        hidden_dim: int | None = None,
        learning_rate: float = 1e-2,
    ) -> None:
        self.n_factors = n_factors
        self.n_epochs = n_epochs
        self.sigma2 = sigma2
        self.beta = beta
        self.hidden_dim = hidden_dim
        self.learning_rate = learning_rate
        self.history: list[float] = []

    def fit(self, train_df: pd.DataFrame) -> None:
        X = ratings_to_matrix(train_df)
        observed = ~np.isnan(X)
        values = np.where(observed, X, 0.0)

        X = torch.tensor(values, dtype=torch.float32)
        mask = torch.tensor(observed, dtype=torch.float32)

        n_users, n_items = X.shape
        k = self.n_factors

        self.user_means = nn.Parameter(torch.randn(n_users, k) * 0.01)
        self.user_logvars = nn.Parameter(torch.zeros(n_users, k))

        if self.hidden_dim is None:
            self.decoder = nn.Linear(k, n_items)

        else:
            self.decoder = nn.Sequential(
                nn.Linear(k, self.hidden_dim),
                nn.Tanh(),
                nn.Linear(self.hidden_dim, n_items),
            )

        # O viés do decodificador começa nas médias dos filmes, e o treino modela o desvio em relação a elas
        counts = np.maximum(observed.sum(axis=0), 1)
        item_means = np.where(observed.any(axis=0), values.sum(axis=0) / counts, values.sum() / observed.sum())
        output_layer = self.decoder if self.hidden_dim is None else self.decoder[-1]
        with torch.no_grad():
            output_layer.bias.copy_(torch.tensor(item_means, dtype=torch.float32))

        parameters = [
            self.user_means,
            self.user_logvars,
            *self.decoder.parameters(),
        ]
        optimizer = torch.optim.Adam(parameters, lr=self.learning_rate)

        self.history = []

        for _ in range(self.n_epochs):
            optimizer.zero_grad()

            std = torch.exp(0.5 * self.user_logvars)
            epsilon = torch.randn_like(std)
            z = self.user_means + std * epsilon

            predictions = self.decoder(z)

            log_likelihood = -0.5 * (
                mask
                * (
                    np.log(2 * np.pi * self.sigma2)
                    + (X - predictions) ** 2 / self.sigma2
                )
            ).sum()

            kl = -0.5 * (
                1
                + self.user_logvars
                - self.user_means**2
                - self.user_logvars.exp()
            ).sum()

            elbo = log_likelihood - self.beta * kl
            loss = -elbo / n_users

            loss.backward()
            optimizer.step()

            self.history.append((-loss.detach()).item())

        with torch.no_grad():
            self.user_embeddings = (
                self.user_means.detach().cpu().numpy().astype(np.float64)
            )
            self.user_variances = (
                self.user_logvars.exp()
                .detach()
                .cpu()
                .numpy()
                .astype(np.float64)
            )

            if self.hidden_dim is None:
                self.item_embeddings = (
                    self.decoder.weight.detach()
                    .cpu().numpy().astype(np.float64)
                )
                self.item_means = (
                    self.decoder.bias.detach()
                    .cpu().numpy().astype(np.float64)
                )

    def _decode_at(self, z: torch.Tensor, items: torch.Tensor) -> torch.Tensor:
        """Decode only the entries (z[..., n, :], items[n]), without computing every movie."""
        if self.hidden_dim is None:
            return (z * self.decoder.weight[items]).sum(-1) + self.decoder.bias[items]
        first, _, last = self.decoder
        return (torch.tanh(first(z)) * last.weight[items]).sum(-1) + last.bias[items]

    def predict(
        self,
        user_id: NDArray[np.intp],
        item_id: NDArray[np.intp],
    ) -> NDArray[np.float64]:
        users = torch.as_tensor(user_id, dtype=torch.long)
        items = torch.as_tensor(item_id, dtype=torch.long)

        with torch.no_grad():
            z = torch.as_tensor(self.user_embeddings, dtype=torch.float32)[users]
            result = self._decode_at(z, items)

        return result.cpu().numpy().astype(np.float64)

    def predict_log_proba(
        self,
        user_id: NDArray[np.intp],
        item_id: NDArray[np.intp],
        ratings: NDArray[np.float64],
        n_samples: int = 100,
        chunk: int = 2048,
    ) -> NDArray[np.float64]:
        """Log of the Monte Carlo predictive density, computed in chunks of rating pairs."""
        users = torch.as_tensor(user_id, dtype=torch.long)
        items = torch.as_tensor(item_id, dtype=torch.long)
        ratings_t = torch.as_tensor(ratings, dtype=torch.float32)
        user_means = torch.as_tensor(self.user_embeddings, dtype=torch.float32)
        user_stds = torch.as_tensor(self.user_variances, dtype=torch.float32).sqrt()

        result = []
        with torch.no_grad():
            for start in range(0, len(users), chunk):
                u = users[start : start + chunk]
                means, stds = user_means[u], user_stds[u]
                z = means + stds * torch.randn(n_samples, *means.shape)
                predictions = self._decode_at(z, items[start : start + chunk])

                log_probs = -0.5 * (
                    np.log(2 * np.pi * self.sigma2)
                    + (ratings_t[start : start + chunk] - predictions) ** 2 / self.sigma2
                )
                result.append(torch.logsumexp(log_probs, dim=0) - np.log(n_samples))

        return torch.cat(result).cpu().numpy().astype(np.float64)

#### Hipótese e previsão — Tarefa 2.2 (VAE linear e não linear)

**Hipótese.** Com decodificador linear $g_\theta(\mathbf z)=\boldsymbol\mu+W\mathbf z$ e o mesmo $k$ e $\sigma^2$ do PPCA, o VAE maximiza um limite inferior do mesmo modelo gerador; a diferença é que o PPCA calcula a posterior de cada usuário de forma exata (EM) e o VAE a aproxima com uma gaussiana diagonal otimizada por gradiente. O decodificador não linear tem mais capacidade, mas cada usuário tem poucas notas (16 a ~600 no treino), então a capacidade extra deve se traduzir em sobreajuste.

**Previsão.** (i) VAE linear e PPCA ficam próximos em RMSE/MAE (diferença menor que o desvio entre folds); o VAE tem NLL ligeiramente pior por causa da posterior diagonal. (ii) O decodificador não linear **não** melhora o RMSE de validação de forma consistente: o erro de treino cai mais do que o de validação (lacuna maior que a do linear). (iii) Em $\beta$: $\beta$ pequeno aproxima um autoencoder determinístico, com sobreajuste; $\beta$ grande colapsa a posterior para o prior e o RMSE tende ao da média por filme. (iv) A curva do ELBO em `history` estabiliza antes do número de épocas escolhido.

**O que refutaria.** Se o decodificador não linear reduzisse o RMSE de validação além do desvio entre folds, ou se o VAE linear ficasse claramente pior que o PPCA em RMSE mesmo com a mesma configuração e convergência.

In [ ]:
K_VAE, SIGMA2_VAE = BEST["ppca_rmse"]["n_factors"], BEST["ppca_rmse"]["sigma2"]   # mesmas suposições do PPCA
print(f"Comparação com o PPCA: k={K_VAE}, σ²={SIGMA2_VAE}")

# --- 1) Convergência (fold 0): curva do ELBO e RMSE de validação em função das épocas ---
tr0, va0 = FOLDS[0]
epochs_list = [100, 200, 400, 800]
conv_rows, histories = [], {}
for label, hidden in [("linear", None), ("não linear (64)", 64)]:
    for n_ep in epochs_list:
        set_seed(SEED)
        m = VAE(n_factors=K_VAE, sigma2=SIGMA2_VAE, beta=1.0, hidden_dim=hidden, n_epochs=n_ep)
        m.fit(tr0)
        met = evaluate_model(m, train=tr0, val=va0)
        conv_rows.append({"decodificador": label, "épocas": n_ep, "train_rmse": met["train_rmse"], "val_rmse": met["val_rmse"], "val_nll": met["val_nll"]})
        if n_ep == max(epochs_list):
            histories[label] = m.history
conv_vae = pd.DataFrame(conv_rows)
print(conv_vae.round(4).to_string(index=False))
lin = conv_vae[conv_vae["decodificador"] == "linear"].set_index("épocas")["val_rmse"]
VAE_EPOCHS = int(next(n for n in epochs_list if lin[n] <= lin.min() + 0.002))
print("\nNº de épocas adotado (RMSE de validação do linear a menos de 0,002 do mínimo):", VAE_EPOCHS)

# --- 2) Validação cruzada ---
fixed = {"n_factors": K_VAE, "sigma2": SIGMA2_VAE, "n_epochs": VAE_EPOCHS}
res_vae_lin = grid_search(VAE, {"beta": [0.1, 0.3, 1.0, 3.0, 10.0]}, fixed={**fixed, "hidden_dim": None}, verbose=False)
res_vae_lin["decodificador"] = "linear"
res_vae_h = grid_search(VAE, {"hidden_dim": [32, 64, 128]}, fixed={**fixed, "beta": 1.0}, verbose=False)
best_h = int(best_row(res_vae_h)["hidden_dim"])
betas_nl = grid_search(VAE, {"beta": [0.1, 0.3, 3.0, 10.0]}, fixed={**fixed, "hidden_dim": best_h}, verbose=False)
res_vae_nl = pd.concat([res_vae_h[res_vae_h["hidden_dim"] == best_h].assign(beta=1.0), betas_nl]).sort_values("beta")
res_vae_nl["decodificador"] = f"não linear ({best_h})"
res_vae = pd.concat([res_vae_lin, res_vae_nl], ignore_index=True)
res_vae.to_csv(RES_DIR / "cv_vae.csv", index=False)
res_vae_h.to_csv(RES_DIR / "cv_vae_hidden.csv", index=False)
BEST["vae_linear"] = {**fixed, "beta": 1.0, "hidden_dim": None}
BEST["vae_nonlinear"] = {**fixed, "beta": 1.0, "hidden_dim": best_h}
print("\nLargura da camada oculta (β=1):\n", res_vae_h[["hidden_dim", "val_rmse", "val_rmse_std", "val_nll"]].round(4).to_string(index=False))
print("\nSensibilidade a β:\n", res_vae[["decodificador", "beta", "val_rmse", "val_rmse_std", "val_mae", "val_nll", "fit_s"]].round(4).to_string(index=False))

# --- 3) Comparação 1: PPCA × VAE linear com as mesmas suposições (validação cruzada) ---
ppca_ref = res_ppca[(res_ppca["n_factors"] == K_VAE) & (res_ppca["sigma2"] == SIGMA2_VAE)].iloc[0]
vae_ref = res_vae_lin[res_vae_lin["beta"] == 1.0].iloc[0]
cmp = pd.DataFrame({"PPCA": ppca_ref[["val_rmse", "val_rmse_std", "val_mae", "val_nll", "fit_s"]],
                    "VAE linear (β=1)": vae_ref[["val_rmse", "val_rmse_std", "val_mae", "val_nll", "fit_s"]]}).astype(float)
print("\nPPCA × VAE linear (mesmos k e σ², validação cruzada):\n", cmp.round(4))

# --- 4) Figuras ---
fig, ax = plt.subplots(1, 4, figsize=(22, 4.5))
for label, hist in histories.items():
    ax[0].plot(hist, label=label)
ax[0].set_title("ELBO médio por usuário (treino, fold 0)"); ax[0].set_xlabel("época"); ax[0].set_ylabel("ELBO"); ax[0].legend()
ax[0].set_ylim(np.percentile(np.concatenate(list(histories.values())), 2), None)
for label, grp in conv_vae.groupby("decodificador"):
    ax[1].plot(grp["épocas"], grp["val_rmse"], marker="o", label=f"{label} (val)")
    ax[1].plot(grp["épocas"], grp["train_rmse"], marker="x", ls="--", label=f"{label} (treino)")
ax[1].set_title("RMSE × épocas (fold 0)"); ax[1].set_xlabel("épocas"); ax[1].legend(fontsize=7)
for label, grp in res_vae.groupby("decodificador"):
    ax[2].errorbar(grp["beta"], grp["val_rmse"], yerr=grp["val_rmse_std"], marker="o", capsize=3, label=label)
    ax[3].errorbar(grp["beta"], grp["val_nll"], yerr=grp["val_nll_std"], marker="o", capsize=3, label=label)
for a, t in [(ax[2], "RMSE de validação × β"), (ax[3], "NLL de validação × β")]:
    a.set_xscale("log"); a.set_title(t); a.set_xlabel("β"); a.legend(fontsize=8)
plt.tight_layout(); plt.savefig(FIG_DIR / "cv_vae.png", dpi=150); plt.show()

# --- 5) Avaliação final no teste ---
for name, cfg in [("VAE linear", BEST["vae_linear"]), ("VAE não linear", BEST["vae_nonlinear"])]:
    out = final_eval(name, VAE(**cfg), train_df, test_df)
    print(name, {k: fmt_ci(v) for k, v in out.items() if k in ("rmse", "mae", "nll")}, f"| treino {out['fit_s']:.1f}s")
print("\nTabela parcial (teste):")
display(final_table())

**Discussão:**
*Escreva suas conclusões aqui*

In [ ]:
# Tabela consolidada das Etapas 1 e 2 (cada modelo foi avaliado uma única vez no teste)
table = final_table()
table.to_csv(RES_DIR / "tabela_final_teste.csv")
display(table)

# RMSE × qualidade das listas: o menor RMSE é a melhor lista?
rows = [{"modelo": n, "rmse": r["rmse"][0], "ndcg@10": r["ndcg@10"][0], "precision@10": r["precision@10"][0]}
        for n, r in FINAL.items() if "rmse" in r]
sc = pd.DataFrame(rows)
pop_row = FINAL["Popularidade (só listas)"]
print("Referência de listas (popularidade): NDCG@10=%.4f | Precision@10=%.4f" % (pop_row["ndcg@10"][0], pop_row["precision@10"][0]))
print("\nPosto por RMSE (1 = melhor):", dict(sc.sort_values("rmse")["modelo"].reset_index(drop=True).rename(lambda i: i + 1)))
print("Posto por NDCG@10 (1 = melhor):", dict(sc.sort_values("ndcg@10", ascending=False)["modelo"].reset_index(drop=True).rename(lambda i: i + 1)))

fig, ax = plt.subplots(1, 3, figsize=(20, 5))
for _, r in sc.iterrows():
    ax[0].scatter(r["rmse"], r["ndcg@10"], s=60)
    ax[0].annotate(r["modelo"], (r["rmse"], r["ndcg@10"]), fontsize=8, xytext=(4, 4), textcoords="offset points")
ax[0].axhline(pop_row["ndcg@10"][0], ls="--", c="gray", label="popularidade"); ax[0].legend()
ax[0].set_xlabel("RMSE (teste)"); ax[0].set_ylabel("NDCG@10"); ax[0].set_title("Erro de nota × qualidade da lista")

names = [n for n in FINAL if n not in ("Média global",)]
ax[1].barh(names, [FINAL[n]["fit_s"] for n in names], color="C0"); ax[1].set_xscale("log")
ax[1].set_title("Tempo de treino (s)"); ax[1].tick_params(labelsize=8)
ax[2].barh(names, [FINAL[n]["predict_s"] for n in names], color="C1"); ax[2].set_xscale("log")
ax[2].set_title("Tempo de inferência no teste (s)"); ax[2].tick_params(labelsize=8)
plt.tight_layout(); plt.savefig(FIG_DIR / "rmse_vs_listas_tempos.png", dpi=150); plt.show()

# RMSE no teste por popularidade do filme
by_pop = pd.DataFrame({n: rmse_by_item_popularity(TRAINED[n], train_df, test_df) for n in FINAL if "rmse" in FINAL[n]})
by_pop.index.name = "nº de notas do filme no treino"
by_pop.to_csv(RES_DIR / "rmse_por_popularidade.csv")
print("\nRMSE no teste por faixa de popularidade do filme:")
display(by_pop.round(4))

---

### Visualização dos espaços latentes

Dos modelos treinados, quatro deles aprendem vetores latentes por usuário e por item: PCA com imputação, fatoração de matrizes, PPCA e VAE.
Esses vetores podem ser analisados com as mesmas ferramentas, permitindo que possamos interpretar e comparar os espaços latentes aprendidos por cada modelo.
Garante-se que pelo menos um modelo de cada tipo tenha sido treinado, e então podemos comparar os espaços latentes aprendidos.

> **Tarefa 2.3**
Visualize os espaços latentes de usuários e de itens dos quatro modelos, usando um $k$ de sua escolha, e relacione o que aparece nas figuras com informações que não foram usadas no treino: por exemplo, gênero, idade e ocupação dos usuários, e gêneros, ano e popularidade dos filmes.
>
> O espaço de cada modelo organiza essas informações, e isso pode ser medido além da inspeção visual?
Os quatro espaços compartilham a mesma estrutura, ou cada suposição produz uma geometria diferente?


**Ideias para começar.**

Para visualizar, projete os vetores em duas dimensões com PCA, que é determinístico e preserva a geometria global, e compare com o t-SNE.
Colorir os pontos por um metadado mostra se ele aparece, e um modelo supervisionado linear (`Ridge` para variáveis contínuas, `LogisticRegression` com `roc_auc_score` para categorias) pode medir quanto de informação sobre o metadado está presente no espaço latente original, sem precisar de uma projeção.

Isso permite verificar se, a partir dos vetores latentes, é possível prever a idade do usuário, o gênero do filme ou a popularidade do filme, quantificando o quanto de informação sobre essas variáveis está presente no espaço latente.

Ao comparar os espaÃ§os latentes, Ã© importante lembrar que cada modelo aprende uma representaÃ§Ã£o diferente, e que a correspondÃªncia entre os vetores latentes de um modelo e de outro nÃ£o Ã© trivial, pois os espaÃ§os latentes podem ser invariantes Ã  rotaÃ§Ã£o, reflexÃ£o e escala.
Para comparar dois espaços é necessário alinhar os vetores latentes de um modelo com os do outro, ou através de análises de correspondência de vizinhos, como a correlação de distâncias entre pontos, ou através da sobreposição dos vizinhos mais próximos de cada filme em dois modelos.

#### Hipótese e previsão — Tarefa 2.3 (espaços latentes)

Comparamos os quatro espaços com o mesmo $k=10$: PCA imputado, fatoração de matrizes (com vieses), PPCA e VAE linear. Como controles, usamos (a) vetores gaussianos aleatórios nas sondagens e (b) a mesma fatoração treinada com **outra semente**, que dá o teto de semelhança esperado quando duas execuções do *mesmo* modelo são comparadas.

**Hipótese.** As informações de metadados não entram no treino, mas as notas dependem delas: usuários de perfis parecidos tendem a avaliar filmes parecidos. Logo, os vetores devem permitir prever, de forma linear e fora da amostra, características correlacionadas com o gosto (gênero e idade do usuário; gênero do filme; ano). A popularidade e a nota média do filme devem ser as variáveis mais fáceis de recuperar, porque estão diretamente ligadas ao padrão de observação e ao nível das notas.

**Previsão.** (i) Todos os modelos superam o controle aleatório (R² ≈ 0, AUC ≈ 0,5) em popularidade e nota média; a recuperação de idade, gênero e ocupação do usuário é fraca a moderada (AUC de gênero ≈ 0,6–0,7), maior para idade que para ocupação. (ii) O PCA imputado concentra a popularidade nos primeiros componentes, pelo efeito da imputação; a fatoração **com vieses** tende a ter *menos* informação de popularidade no espaço latente, porque o viés do filme a absorve. (iii) Os espaços de PPCA e VAE linear são mais parecidos entre si (mesma família, mesmo prior) do que com o PCA imputado, e a semelhança entre os modelos fica abaixo do teto dado por duas sementes da mesma fatoração. (iv) A projeção por t-SNE mostra agrupamentos mais "bonitos" que a PCA, mas as sondagens lineares é que medem o conteúdo de fato.

**O que refutaria.** Se os controles aleatórios tivessem AUC/R² semelhantes aos modelos, o espaço não carregaria informação dos metadados; se duas sementes da mesma fatoração fossem tão diferentes quanto modelos distintos, as diferenças entre modelos seriam ruído de inicialização.

In [ ]:
K_LAT = 10
set_seed(SEED)

# ---------- 1) Treina os quatro modelos com o mesmo k (e a mesma fatoração com outra semente) ----------
def fit_space(name, model, seed=SEED):
    set_seed(seed)
    model.fit(train_df)
    return {"user": np.asarray(model.user_embeddings), "item": np.asarray(model.item_embeddings)}

SPACES = {
    "PCA imputado": fit_space("pca", ImputedPCA(n_components=K_LAT)),
    "MF": fit_space("mf", FastMatrixFactorization(n_factors=K_LAT, n_epochs=MF_EPOCHS, reg=BEST["mf_bias"]["reg"], bias_reg=BEST["mf_bias"]["bias_reg"])),
    "PPCA": fit_space("ppca", FastPPCA(n_factors=K_LAT, sigma2=BEST["ppca_rmse"]["sigma2"], n_epochs=PPCA_EPOCHS)),
    "VAE linear": fit_space("vae", VAE(n_factors=K_LAT, sigma2=SIGMA2_VAE, beta=1.0, n_epochs=VAE_EPOCHS)),
}
CONTROLS = {
    "MF (outra semente)": fit_space("mf2", FastMatrixFactorization(n_factors=K_LAT, n_epochs=MF_EPOCHS, reg=BEST["mf_bias"]["reg"], bias_reg=BEST["mf_bias"]["bias_reg"]), seed=SEED + 1),
}
rng = np.random.default_rng(SEED)
RANDOM_SPACE = {"user": rng.normal(size=(N_USERS, K_LAT)), "item": rng.normal(size=(N_ITEMS, K_LAT))}

# ---------- 2) Metadados (não usados no treino) ----------
item_n_train = np.bincount(train_df["item_id"], minlength=N_ITEMS)
item_mean_train = train_df.groupby("item_id")["rating"].mean().reindex(range(N_ITEMS)).to_numpy()
user_meta = {"idade": users["age"].to_numpy(dtype=float), "gênero (M=1)": (users["gender"] == "M").to_numpy(int)}
occ_codes = pd.Categorical(users["occupation"]).codes
item_meta_reg = {"ano": movies["year"].to_numpy(dtype=float), "log nº de notas": np.log1p(item_n_train), "nota média (treino)": item_mean_train}
genre_ok = [g for g in GENRES if movies[g].sum() >= 30]      # gêneros com >= 30 filmes
print("Gêneros usados nas sondagens:", genre_ok)

# ---------- 3) Sondagens lineares sem vazamento: validação cruzada de 5 folds, padronização dentro do fold ----------
cv_kf = KFold(5, shuffle=True, random_state=SEED)
cv_skf = StratifiedKFold(5, shuffle=True, random_state=SEED)

def probe_reg(Z, y):
    ok = ~np.isnan(y)
    pipe = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 12)))
    s = cross_val_score(pipe, Z[ok], y[ok], cv=cv_kf, scoring="r2")
    return s.mean(), s.std(ddof=1)

def probe_clf(Z, y, multiclass=False):
    pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000))
    s = cross_val_score(pipe, Z, y, cv=cv_skf, scoring="roc_auc_ovr" if multiclass else "roc_auc")
    return s.mean(), s.std(ddof=1)

def probe_user_space(Z):
    out = {"idade (R²)": probe_reg(Z, user_meta["idade"]), "gênero (AUC)": probe_clf(Z, user_meta["gênero (M=1)"]),
           "ocupação (AUC macro)": probe_clf(Z, occ_codes, multiclass=True)}
    return out

def probe_item_space(Z):
    out = {f"{k} (R²)": probe_reg(Z, v) for k, v in item_meta_reg.items()}
    per_genre = {g: probe_clf(Z, movies[g].to_numpy(int)) for g in genre_ok}
    out["gêneros (AUC macro)"] = (np.mean([v[0] for v in per_genre.values()]), np.mean([v[1] for v in per_genre.values()]))
    return out, per_genre

all_spaces = {**SPACES, **CONTROLS, "Aleatório (controle)": RANDOM_SPACE}
probe_users = pd.DataFrame({n: {k: f"{m:.3f} ± {s:.3f}" for k, (m, s) in probe_user_space(sp["user"]).items()} for n, sp in all_spaces.items()})
item_probe, genre_probe = {}, {}
for n, sp in all_spaces.items():
    item_probe[n], genre_probe[n] = probe_item_space(sp["item"])
probe_items = pd.DataFrame({n: {k: f"{m:.3f} ± {s:.3f}" for k, (m, s) in d.items()} for n, d in item_probe.items()})
probe_genres = pd.DataFrame({n: {g: v[0] for g, v in d.items()} for n, d in genre_probe.items()}).round(3)
probe_users.to_csv(RES_DIR / "sondagem_usuarios.csv"); probe_items.to_csv(RES_DIR / "sondagem_filmes.csv"); probe_genres.to_csv(RES_DIR / "sondagem_generos.csv")
print("Sondagem linear dos USUÁRIOS (média ± desvio entre folds):"); display(probe_users)
print("Sondagem linear dos FILMES:"); display(probe_items)
print("AUC por gênero (média dos folds):"); display(probe_genres)

# ---------- 4) Projeções 2D (PCA e t-SNE) ----------
PROJ = {}
def project(name, entity, method):
    key = (name, entity, method)
    if key not in PROJ:
        Z = SPACES[name][entity]
        PROJ[key] = PCA(2, random_state=SEED).fit_transform(Z) if method == "PCA" else \
            TSNE(2, perplexity=30, init="pca", random_state=SEED).fit_transform(Z)
    return PROJ[key]

main_genre = np.select([movies[g].to_numpy(bool) for g in ["Horror", "Action", "Comedy", "Drama"]], ["Horror", "Action", "Comedy", "Drama"], default="Outros")
COLORINGS = {
    "user": {"idade": (user_meta["idade"], "cont"), "gênero": (np.where(user_meta["gênero (M=1)"] == 1, "M", "F"), "cat")},
    "item": {"log nº de notas": (item_meta_reg["log nº de notas"], "cont"), "ano": (item_meta_reg["ano"], "cont"),
             "gênero principal": (main_genre, "cat")},
}

def plot_projections(entity, label):
    values, kind = COLORINGS[entity][label]
    fig, axes = plt.subplots(2, len(SPACES), figsize=(4.6 * len(SPACES), 8.5))
    for j, name in enumerate(SPACES):
        for i, method in enumerate(["PCA", "t-SNE"]):
            P = project(name, entity, method)
            ax = axes[i, j]
            if kind == "cont":
                sc_ = ax.scatter(P[:, 0], P[:, 1], c=values, s=7, cmap="viridis")
            else:
                for cat in pd.unique(values):
                    m = values == cat
                    ax.scatter(P[m, 0], P[m, 1], s=7, alpha=0.7, label=cat)
            ax.set_title(f"{name} — {method}", fontsize=10); ax.set_xticks([]); ax.set_yticks([])
    if kind == "cont":
        fig.colorbar(sc_, ax=axes, shrink=0.6, label=label)
    else:
        axes[0, -1].legend(markerscale=3, fontsize=8)
    fig.suptitle(f"{'Usuários' if entity == 'user' else 'Filmes'} coloridos por {label} (informação não usada no treino)", y=1.0)
    plt.savefig(FIG_DIR / f"proj_{entity}_{label.replace(' ', '_')}.png", dpi=130, bbox_inches="tight"); plt.show()

for entity, labels in COLORINGS.items():
    for label in labels:
        plot_projections(entity, label)

# ---------- 5) Comparação entre espaços, sem precisar alinhar: correlação de distâncias, vizinhos (Jaccard) e CKA ----------
from sklearn.metrics import pairwise_distances

def linear_cka(X, Y):
    X, Y = X - X.mean(0), Y - Y.mean(0)
    return float(np.linalg.norm(Y.T @ X, "fro") ** 2 / (np.linalg.norm(X.T @ X, "fro") * np.linalg.norm(Y.T @ Y, "fro")))

def knn_sets(D, k=10):
    D = D.copy(); np.fill_diagonal(D, np.inf)
    nn = np.argpartition(D, k, axis=1)[:, :k]
    mask = np.zeros(D.shape, dtype=bool); mask[np.arange(len(D))[:, None], nn] = True
    return mask

def compare_spaces(entity, k_nn=10):
    names = list(SPACES) + list(CONTROLS)
    embs = {n: {**SPACES, **CONTROLS}[n][entity] for n in names}
    iu = np.triu_indices(len(next(iter(embs.values()))), 1)
    dist = {m: {n: pairwise_distances(Z, metric=m) for n, Z in embs.items()} for m in ["euclidean", "cosine"]}
    knn = {n: knn_sets(dist["cosine"][n], k_nn) for n in names}
    mats = {key: pd.DataFrame(np.eye(len(names)), index=names, columns=names) for key in ["Spearman (eucl.)", "Spearman (cos.)", f"Jaccard {k_nn}-NN (cos.)", "CKA linear"]}
    for a, b in product(names, names):
        if a == b:
            continue
        mats["Spearman (eucl.)"].loc[a, b] = spearmanr(dist["euclidean"][a][iu], dist["euclidean"][b][iu])[0]
        mats["Spearman (cos.)"].loc[a, b] = spearmanr(dist["cosine"][a][iu], dist["cosine"][b][iu])[0]
        inter = (knn[a] & knn[b]).sum(1)
        mats[f"Jaccard {k_nn}-NN (cos.)"].loc[a, b] = np.mean(inter / (2 * k_nn - inter))
        mats["CKA linear"].loc[a, b] = linear_cka(embs[a], embs[b])
    return mats

for entity, title in [("item", "Filmes"), ("user", "Usuários")]:
    mats = compare_spaces(entity)
    fig, axes = plt.subplots(1, 4, figsize=(24, 5))
    for ax, (key, M) in zip(axes, mats.items()):
        sns.heatmap(M.astype(float), annot=True, fmt=".2f", cmap="viridis", vmin=0, vmax=1, ax=ax, cbar=False)
        ax.set_title(f"{title}: {key}"); ax.tick_params(labelsize=8)
    plt.tight_layout(); plt.savefig(FIG_DIR / f"comparacao_espacos_{entity}.png", dpi=130); plt.show()
    pd.concat(mats, names=["medida"]).to_csv(RES_DIR / f"comparacao_espacos_{entity}.csv")

# Norma do vetor do filme × popularidade (a popularidade se reflete na escala?)
norms = pd.DataFrame({n: [spearmanr(np.linalg.norm(sp["item"], axis=1), item_meta_reg["log nº de notas"])[0],
                          spearmanr(np.linalg.norm(sp["item"], axis=1), item_meta_reg["nota média (treino)"], nan_policy="omit")[0]]
                      for n, sp in SPACES.items()}, index=["ρ(norma, log nº notas)", "ρ(norma, nota média)"])
print("\nCorrelação de Spearman entre a norma do vetor do filme e a popularidade / nota média:"); display(norms.round(3))

**Discussão:**
*Escreva suas conclusões aqui*

---

## Etapa 3: Projeto do grupo

Nesta última etapa, cada um de vocês deve propor um projeto de investigação, que pode ser uma melhoria do modelo ou uma análise mais profunda de algum aspecto do problema.
Após finalizar as duas primeiras etapas, verifiquem se há alguma pergunta que surgiu durante as análises e ainda não foi explorado, ou alguma hipótese que vocês gostariam de testar, por exemplo, sobre a geometria do espaço latente, uma formulação alternativa, ou um hiperparâmetro que não foi isolado.

O importante é que o projeto esteja bem definido, com uma motivação clara, uma hipótese testável e um protocolo experimental que permita avaliar o impacto da mudança proposta.

> **Tarefa 3.1**
Escolha um projeto e escreva a pergunta, a hipótese, a previsão e o experimento antes de rodar.
Defina o modelo referência, qual é a mudança proposta e qual resultado refutaria a hipótese de vocês.
Ao final, compare o que foi previsto com o que foi observado e o que o resultado diz sobre a previsão sobre o espaço latente.

**Projetos sugeridos.**
Aqui está uma lista de ideias que podem ser exploradas, mas vocês podem propor outras, desde que estejam bem justificadas.

*1. Ausência não aleatória.*
Usuários ativos e filmes populares aparecem mais nos dados, o que cria um viés de observação.
Pergunta: corrigir esse viés com pesos pelo inverso da probabilidade de observação [Schnabel et al.](https://arxiv.org/abs/1602.05352) melhora as previsões para filmes pouco avaliados, e o que acontece com os populares?

*2. Metadados como features.*
Ao invés de tratar cada usuário e item como um identificador único, podemos usar suas características como features para o "codificador".
Pergunta: usar os metadados para definir a média do prior de $\mathbf{z}$ melhora as previsões de usuários com pouco histórico?

*3. Geometria da compatibilidade.*
Todos os modelos utilizam o produto interno para medir a compatibilidade entre usuários e itens.
Pergunta: trocar o produto interno por uma distância, ou por uma pequena rede que recebe $\mathbf{z}_u$ e $\mathbf{w}_i$, muda o que o espaço codifica e o erro de previsão? Ver [Rendle et al.](https://arxiv.org/abs/2005.09683) para um resultado que motiva a pergunta.

*4. Estrutura do espaço latente.*
A selecão de hiperparâmetros não observou a geometria do espaço latente, mas apenas o erro de previsão.
Pergunta: o que muda no espaço latente aprendido do VAE ao variar $\beta$, o tamanho do espaço ou a regularização, em termos de estabilidade de clusters, correlação com metadados e erro de previsão?

## Tarefa 3.1 — Projeto do grupo (Sugestão D): estrutura do espaço latente no VAE

> Esta seção foi escrita **antes** de rodar o experimento.

**1. A pergunta.** O que muda no espaço latente do VAE linear quando variamos $\beta$ (peso do KL), $\sigma^2$ e a dimensão $k$? Em particular: (a) quantas dimensões ficam ativas; (b) quão estáveis são os agrupamentos de usuários e as vizinhanças de filmes entre sementes; (c) quanta informação de metadados (idade e gênero do usuário; ano, popularidade e gênero do filme) é recuperável linearmente; (d) como o erro de previsão (RMSE) e o NLL acompanham essas mudanças. A seleção de hiperparâmetros da Etapa 2 olhou só para o erro de previsão; aqui olhamos também para a geometria.

**2. A hipótese (mecanismo).** Multiplicando o ELBO por $\sigma^2$, o objetivo vira $-\tfrac12\sum(r_{ui}-\hat r_{ui})^2-\beta\sigma^2\,\mathrm{KL}$. Logo, para o decodificador linear, só o produto $\gamma=\beta\sigma^2$ importa para as **médias** aprendidas: $\beta$ e $\sigma^2$ não são botões independentes para a estrutura do espaço, e $\sigma^2$ só muda a escala do NLL. Um $\gamma$ grande empurra a posterior de cada usuário para o prior $\mathcal N(\mathbf 0,I)$; nas dimensões em que a verossimilhança traz pouca informação (usuários com poucas notas), o ganho de reconstrução não paga o KL e a dimensão **colapsa** (variância das médias entre usuários ≈ 0). Dimensões colapsadas não carregam informação e, como o decodificador não é treinado nelas, injetam ruído nas vizinhanças dos filmes.

**3. O experimento.** Modelo de referência: VAE linear com $k=20$, $\sigma^2=1$, $\beta=1$, treinado no fold 0 (80% do treino) e avaliado no restante (validação); o teste não é usado na análise, apenas em uma checagem final de três valores de $\beta$. Variações, uma por vez, com 3 sementes cada:
- $\beta\in\{0{,}01;\,0{,}03;\,0{,}1;\,0{,}3;\,1;\,3;\,10\}$ (com $k=20$, $\sigma^2=1$);
- $k\in\{2,5,10,20,50\}$ (com $\beta=1$, $\sigma^2=1$);
- pares com o mesmo $\gamma=\beta\sigma^2$ e $\sigma^2\in\{0{,}5;1;2\}$, para $\gamma\in\{0{,}3;\,1\}$.

Métricas: RMSE e NLL de validação; nº de dimensões ativas (variância entre usuários das médias $\mathbf m_u$ maior que 0,01); sondagens lineares com validação cruzada (idade R², gênero AUC dos usuários; ano R², popularidade R² e AUC médio dos gêneros dos filmes); estabilidade entre sementes: ARI de KMeans ($8$ grupos) nos usuários e sobreposição de Jaccard dos 10 vizinhos mais próximos dos filmes.

**4. As previsões.**
- **P1 (colapso).** Dimensões ativas diminuem monotonicamente com $\beta$; em $\beta=1$, $k=20$ menos da metade das dimensões fica ativa; em $\beta=10$, no máximo 1–3.
- **P2 (erro).** O RMSE de validação tem mínimo em $\beta$ intermediário (entre 0,1 e 1) e sobe para $\beta\ge3$ rumo ao erro da média por filme (~1,0); para $\beta\le0{,}03$ há sobreajuste (RMSE pior que o mínimo e NLL muito pior).
- **P3 (metadados).** A recuperação de gênero e idade dos usuários é máxima para $\beta$ intermediário: com $\beta$ muito pequeno os vetores carregam ruído idiossincrático; com $\beta$ muito grande o espaço colapsa.
- **P4 (estabilidade).** A estabilidade entre sementes (ARI, Jaccard) **cresce** com $\beta$, porque o prior remove direções arbitrárias e deixa as direções de maior variância (como no PCA).
- **P5 (invariância).** Para o mesmo $\gamma$ e a mesma semente, RMSE, dimensões ativas e sondagens diferem entre $\sigma^2$ distintos por muito menos que o desvio entre sementes; só o NLL muda.
- **P6 ($k$).** Com $\beta=1$, aumentar $k$ além de ~10 não melhora o RMSE: as dimensões extras colapsam e o número de dimensões ativas satura.

**5. O que refutaria.**
- P1: dimensões ativas não decrescentes em $\beta$, ou ≥ 10 ativas em $\beta=10$.
- P2: RMSE monotonicamente decrescente em $\beta$ (sem mínimo interno) ou monotonicamente crescente.
- P3: o máximo da recuperação de gênero/idade em um extremo da grade, fora da faixa de ruído entre sementes.
- P4: ARI/Jaccard sem tendência de crescimento com $\beta$ (diferença menor que o desvio entre sementes).
- P5: diferenças entre $\sigma^2$ no mesmo $\gamma$ maiores que o desvio entre sementes.
- P6: RMSE de $k=50$ significativamente melhor que o de $k=10$, ou dimensões ativas crescendo proporcionalmente a $k$.

In [ ]:
SEEDS = [0, 1, 2]
tr0, va0 = FOLDS[0]                                           # treino/validação do experimento (o teste não é usado aqui)
present_items = np.where(np.bincount(tr0["item_id"], minlength=N_ITEMS) > 0)[0]
N_STUDY_EPOCHS = VAE_EPOCHS

def user_probe(zu):
    return {"idade_R2": probe_reg(zu, user_meta["idade"])[0], "genero_AUC": probe_clf(zu, user_meta["gênero (M=1)"])[0]}

def item_probe_fn(wi, ids):
    ids = ids[ids < len(wi)]
    Z = wi[ids]
    out = {"ano_R2": probe_reg(Z, item_meta_reg["ano"][ids])[0],
           "logpop_R2": probe_reg(Z, item_meta_reg["log nº de notas"][ids])[0]}
    out["generos_AUC"] = float(np.mean([probe_clf(Z, movies[g].to_numpy(int)[ids])[0] for g in genre_ok]))
    return out, ids

def run_config(k, beta, sigma2, seed):
    set_seed(seed)
    m = VAE(n_factors=k, sigma2=sigma2, beta=beta, n_epochs=N_STUDY_EPOCHS)
    m.fit(tr0)
    met = evaluate_model(m, val=va0)
    zu, wi = m.user_embeddings, m.item_embeddings
    s2 = m.user_variances
    kl_dim = 0.5 * (s2 + zu ** 2 - 1 - np.log(s2)).mean(0)            # KL médio por dimensão
    ip, ids = item_probe_fn(wi, present_items)
    row = {"k": k, "beta": beta, "sigma2": sigma2, "gamma": beta * sigma2, "seed": seed,
           "val_rmse": met["val_rmse"], "val_nll": met["val_nll"],
           "dim_ativas": int((zu.var(0) > 0.01).sum()), "kl_total": float(kl_dim.sum()),
           **user_probe(zu), **ip}
    return row, zu, wi[ids], ids

def stability(embs):
    """ARI (KMeans, 8 grupos) entre usuários e Jaccard dos 10-NN dos filmes, para todos os pares de sementes."""
    km = {s: KMeans(8, n_init=10, random_state=SEED).fit_predict(zu) for s, (zu, _, _) in embs.items()}
    nn = {s: knn_sets(pairwise_distances(wi, metric="cosine"), 10) for s, (_, wi, _) in embs.items()}
    aris, jac, ckas = [], [], []
    for a, b in [(0, 1), (0, 2), (1, 2)]:
        aris.append(adjusted_rand_score(km[a], km[b]))
        inter = (nn[a] & nn[b]).sum(1)
        jac.append(np.mean(inter / (20 - inter)))        # união = 2·10 − interseção
        ckas.append(linear_cka(embs[a][0], embs[b][0]))
    return {"ari_usuarios": np.mean(aris), "jaccard_filmes": np.mean(jac), "cka_usuarios": np.mean(ckas)}

configs = ([(20, b, 1.0) for b in [0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0]] +
           [(k, 1.0, 1.0) for k in [2, 5, 10, 50]] +
           [(20, 0.6, 0.5), (20, 0.15, 2.0), (20, 2.0, 0.5), (20, 0.5, 2.0)])
rows, stab = [], []
for (k, beta, sigma2) in configs:
    embs = {}
    for seed in SEEDS:
        row, zu, wi, ids = run_config(k, beta, sigma2, seed)
        rows.append(row); embs[seed] = (zu, wi, ids)
    stab.append({"k": k, "beta": beta, "sigma2": sigma2, **stability(embs)})
    r = pd.DataFrame(rows[-len(SEEDS):])
    print(f"k={k:>2} β={beta:<5} σ²={sigma2:<3} | RMSE={r.val_rmse.mean():.4f} NLL={r.val_nll.mean():.3f} ativas={r.dim_ativas.mean():.1f} "
          f"gênero AUC={r.genero_AUC.mean():.3f} ARI={stab[-1]['ari_usuarios']:.3f} Jaccard={stab[-1]['jaccard_filmes']:.3f}", flush=True)

runs = pd.DataFrame(rows)
stab = pd.DataFrame(stab)
runs.to_csv(RES_DIR / "etapa3_execucoes.csv", index=False); stab.to_csv(RES_DIR / "etapa3_estabilidade.csv", index=False)
metrics = ["val_rmse", "val_nll", "dim_ativas", "kl_total", "idade_R2", "genero_AUC", "ano_R2", "logpop_R2", "generos_AUC"]
agg = runs.groupby(["k", "beta", "sigma2"])[metrics].agg(["mean", "std"])
agg.columns = [f"{a}_{b}" for a, b in agg.columns]
summary = agg.reset_index().merge(stab, on=["k", "beta", "sigma2"])
summary.to_csv(RES_DIR / "etapa3_resumo.csv", index=False)

# ---------- Figuras: efeito de β (k=20, σ²=1) e de k (β=1, σ²=1) ----------
panels = [("val_rmse", "RMSE de validação"), ("val_nll", "NLL de validação"), ("dim_ativas", "dimensões ativas"),
          ("genero_AUC", "gênero do usuário (AUC)"), ("idade_R2", "idade do usuário (R²)"), ("generos_AUC", "gêneros dos filmes (AUC médio)"),
          ("logpop_R2", "popularidade (R²)"), ("ari_usuarios", "ARI dos grupos de usuários entre sementes"), ("jaccard_filmes", "Jaccard 10-NN dos filmes entre sementes")]
for factor, sel, fname in [("beta", (summary["k"] == 20) & (summary["sigma2"] == 1.0), "etapa3_beta"),
                           ("k", (summary["beta"] == 1.0) & (summary["sigma2"] == 1.0), "etapa3_k")]:
    sub = summary[sel].sort_values(factor)
    fig, axes = plt.subplots(3, 3, figsize=(16, 11))
    for ax, (col, title) in zip(axes.ravel(), panels):
        mean = sub[col + "_mean"] if col + "_mean" in sub else sub[col]
        std = sub[col + "_std"] if col + "_std" in sub else None
        ax.errorbar(sub[factor], mean, yerr=std, marker="o", capsize=3)
        ax.set_xscale("log"); ax.set_title(title, fontsize=10); ax.set_xlabel(factor)
    plt.tight_layout(); plt.savefig(FIG_DIR / f"{fname}.png", dpi=140); plt.show()

# ---------- P5: mesmo γ = β·σ², σ² diferentes ----------
summary["gamma"] = (summary["beta"] * summary["sigma2"]).round(3)
iso = summary[summary["gamma"].isin([0.3, 1.0]) & (summary["k"] == 20)].sort_values(["gamma", "sigma2"])
print("\nP5 — mesmo γ=β·σ², σ² diferente (média ± desvio entre 3 sementes):")
display(iso[["gamma", "beta", "sigma2", "val_rmse_mean", "val_rmse_std", "val_nll_mean", "dim_ativas_mean", "genero_AUC_mean", "genero_AUC_std", "generos_AUC_mean", "ari_usuarios"]].round(4))
# mesma semente: diferença pareada entre σ² para o mesmo γ
pair = runs.assign(gamma=(runs["beta"] * runs["sigma2"]).round(3)).query("k == 20 and gamma in [0.3, 1.0]")
paired = pair.pivot_table(index=["gamma", "seed"], columns="sigma2", values="val_rmse")
print("\nRMSE de validação por semente (colunas = σ²):"); display(paired.round(4))
print("Desvio entre sementes (β=1, σ²=1):", round(runs.query("k == 20 and beta == 1.0 and sigma2 == 1.0")["val_rmse"].std(ddof=1), 5))

# ---------- Checagem final no TESTE (uma vez por modelo): três valores de β com k=20, σ²=1 ----------
for beta in [0.1, 1.0, 10.0]:
    out = final_eval(f"β-VAE (k=20, β={beta:g})", VAE(n_factors=20, sigma2=1.0, beta=beta, n_epochs=N_STUDY_EPOCHS), train_df, test_df)
    print(f"β={beta:<5g}", {k: fmt_ci(v) for k, v in out.items() if k in ("rmse", "mae", "nll", "ndcg@10")})

**Discussão:**
*Escreva suas conclusões aqui*

In [ ]:
# Empacota figuras e tabelas para levar do Colab para o repositório
import shutil

export = Path("export")
shutil.rmtree(export, ignore_errors=True)
shutil.copytree(FIG_DIR, export / "figs")
shutil.copytree(RES_DIR, export / "results")
shutil.make_archive("resultados_mc847", "zip", export)
print("Arquivo gerado:", Path("resultados_mc847.zip").resolve())
try:
    from google.colab import files
    files.download("resultados_mc847.zip")
except ImportError:
    print("Fora do Colab: o arquivo fica na pasta atual.")